# Macaque MRI Atlas — TEST lightweight (v22)

One slice, end to end. Registration (**2.1–2.3**) is read from the cache
`REGISTRATION_ONLY_v4` wrote; nothing here re-registers.
Every tunable value lives in cell 1 (`0 SETUP AND PARAMETERS`); no later cell assigns one.

| cell | tree node |
|---|---|
| 1 | 0.1–0.4 setup + all parameters |
| 2 | 1.2 stage scans |
| 3 | 2.x load registration cache |
| 4 | 3.1 parcellation |
| 5 | 3.2 / 4.4 SVG round-trip |
| 6 | 5.8 perturbation harness |
| 7 | 5.1–5.8 contour fusion |
| 8 | 5.7.6 render |
| 9 | 6.1 mirror back |
| 10 | 5.8.3 / 5.8.4 verification |
| 11 | 5.7.4 region registry |


In [ ]:
# =====================================================================================
# 0  SETUP AND PARAMETERS
# =====================================================================================
# Lightweight TEST notebook: runs 3.1 -> 5.8 on ONE slice, reusing the registration
# products REGISTRATION_ONLY wrote (nodes 2.1-2.3 are read from cache, never recomputed).
# EVERY tunable value in this notebook is set in this cell. No later cell assigns one.
from google.colab import drive; drive.mount('/content/drive')
!pip -q install antspyx nibabel numpy scipy shapely matplotlib scikit-image svgpathtools

import os, glob, json, time, pickle, subprocess, sys, csv, colorsys, re
import numpy as np
import nibabel as nib
import ants
from pathlib import Path

# =====================================================================================
# 0.4.1  PATHS AND RUN IDENTITY
# =====================================================================================
DRIVE_ROOT  = "/content/drive/My Drive/macaque_atlas"
WORK        = f"{DRIVE_ROOT}/work_TESTv5"   # read-only: source of the 2.x registration cache
WORK_TEST   = f"{DRIVE_ROOT}/work_TESTv5"   # parent of the per-run output tree
MODULES_DIR = f"{DRIVE_ROOT}/modules"       # boundary_graph.py, edge_fusion.py, contour_fusion.py
TPL_DIR     = f"{DRIVE_ROOT}/template"
TEMPLATE    = "DB09"
PARCELLATION_LEVEL = "full"
SLICE_AXIS  = 1        # coronal; must match REGISTRATION_ONLY
RUN_PREFIX  = "test_perturb"   # name of the sub-folder under WORK_TEST this run writes into

# =====================================================================================
# 0.4.2  SUBJECTS                                                                  [1.2.1]
# =====================================================================================
# Must be byte-identical to the same block in REGISTRATION_ONLY, FULL_PIPELINE and the
# Experiment Runner. Everything downstream loops over whatever is here.
SUBJECT_SCANS = {
    "NHP1": f"{DRIVE_ROOT}/scans/nifti_out/NHP1/NHP1_scan9_reco1_zfix_skullstripped_cropped_final.nii.gz",
    "NHP2": f"{DRIVE_ROOT}/scans/nifti_out/NHP2/NHP2_scan3_reco1_skullstripped_cropped_final.nii.gz",
}

# =====================================================================================
# 0.4.4  PARCELLATION                                                            [3.1.x]
# =====================================================================================
SLICE_INDEX       = 50          # 3.1.7  which coronal slice to test on (subject grid)
SLICE_RANGE       = (12, 78)    # 3.1.7  advisory guard-rail for SLICE_INDEX
HEMISPHERE        = "left"      # 3.1.1  "whole" | "left" | "right"
MIN_REGION_VOXELS = 6           # 3.1.3.1  regions smaller than this are dropped before tracing
SIMPLIFY_TOL      = 0.25        # 3.1.4.2  px. Douglas-Peucker tolerance: the fidelity knob.
SMOOTH_BOUNDARIES = True        # 3.1.4.1  False = exact pixel staircase
SMOOTH_SIGMA      = 2.0         # 3.1.4.1  px. Gaussian low-pass on the traced boundary.
TRACE_MAX_HANDLES = None        # 3.1.4.3  None = keep the DP vertices; an int caps handles LOSSILY
SEED_MIN_VOXELS   = 6           # 3.1.6.1  a region needs this many voxels to get a seed

# =====================================================================================
# 0.4.5  EXPORT AND QA                                                       [3.2.x, 4.4]
# =====================================================================================
LABEL_DETAIL       = "abbrev"   # 3.2.3.3  "none" | "abbrev" | "full"
LINE_MODE          = "single"   # 3.2.3.4  "single" | "double"
LINE_DOUBLE_OFFSET = 0.35       # 3.2.3.4  voxel offset of each half-line (double mode only)
FLIP_LR            = False      # 3.2.1   horizontal flip of the exported view only
LABEL_ALL_INSTANCES  = True     # 3.2.3.3  label every fragment, not just the largest
MIN_COMPONENT_VOXELS = 8        # 3.2.3.3  a component smaller than this gets no label
LABEL_FIT          = 0.60       # 3.2.3.3  text must fit this fraction of the component width
LABEL_HEIGHT_FRAC  = 0.9        # 3.2.3.3
LABEL_FS_MAX       = 2.2        # 3.2.3.3  px
LABEL_FS_MIN       = 0.55       # 3.2.3.3  px
LABEL_OUTLINE_FRAC = 0.16       # 3.2.3.3
EXPORT_SMOOTH_SIGMA = 0.0       # 3.2.3.2  0 = export the traced geometry unchanged
RENDER_SMOOTH_ITERS = 2         # 3.2.3.2  display-only corner rounding
LEGEND_GAP = 4.0; LEGEND_FS = 2.0; LEGEND_ROW = 2.6; LEGEND_SW = 2.0   # 3.2.4
AUTO_QA_COPY = True             # 3.2.7  copy the export into RUN_DIR/qc/ for fusion input

# =====================================================================================
# 0.4.6  FUSION                                                                    [5.x]
# =====================================================================================
# Units are VOXELS; 1 voxel == 1 SVG px.
# Tuning order: kp_alpha (scale of preserved detail) -> fit_tol_px (rendering fidelity)
# -> node_match_max (from the measured S3_node_disp_p95_px) -> curv_lambda / kp_gamma.
# kp_alpha COARSER than the smallest feature is worse than keypoints=False: the greedy
# alpha-separation rule keeps only one of a notch's two tip corners and pulls it apart.
FUSION_PARAMS_KW = dict(
    # 5.6.4  anchors (F1 / SATM Step 1+2)
    keypoints      = False,
    kp_alpha       = 4.0,     # px. Min spacing between key points.
    kp_curv_thresh = 0.05,    # 1/px. Both signs kept; -ve = concavity (SATM CURV).
    kp_dmax        = 5.0,     # px. Beyond this, key points are rejected, not force-matched.
    kp_gamma       = 1/3,     # 5.6.4.2.1  weight on the relative arc-length coordinate
    kp_l_scale     = None,    # None -> mean arc length, so gamma*l is in px

    # 5.6.6.1  sampling (F1a)
    fit_tol_px  = 0.05,       # max chord deviation; drives the point count (sagitta bound)
    n_min_seg   = 2,
    n_max_seg   = 200,        # clamp; saturation is reported as S7_budget_saturated
    curv_lambda = 0.5,        # 5.6.6.2  0 = arc-length, 1 = pure curvature

    # 5.6.3  curvature scale. None -> min(1.0, kp_alpha/4); must stay finer than kp_alpha.
    curv_sigma_px  = None,
    curv_sample_px = 0.25,

    # 5.5  correspondence (F2, F3)
    node_tol        = 1.0,    # 5.2.2.1  within a subject: endpoints this close are one junction
    node_match_max  = 35.0,   # 5.5.1.2  across subjects: refuse to pair junctions further apart
    strict_topo     = False,  # 5.5.3.1  region-adjacency divergence is a hard stop
    strict_topo_min = 0.98,

    # 5.6.1  policy (F4)
    orphan_policy   = "passthrough",   # passthrough | drop | reference
    min_arc_support = 1,
    reference_sid   = "AUTO",  # 5.6.1.1  a subject id forces the reference; AUTO uses the ladder

    # 5.3  phantom regions
    phantom_eps_px     = 1e-3,  # 5.3.3.1.2  length given to a POINT phantom so its arcs survive
    halt_on_situation3 = True,  # 5.3.3.4.3  Situation 3 refuses the slice even if strict_topo off
    label_all_fragments = True, # 5.6.4.4  anchors on every fragment, not just the largest

    # 5.6.2  representation (F1b)
    representation = "polyline",   # "spline" fits per segment, so corners survive
    spline_smooth  = 0.0,
    loop_align     = "fft",        # 5.6.2.1  "fft" (exact optimum) | "coarse"

    # 5.7  rebuild (F5)
    outer_code         = 0,      # 5.7.3.1  DB09 background
    unlabeled_id_start = 8001,   # 5.7.3.4  an undecidable face becomes an explicit UNLABELED id
    new_id_start       = 9001,   # 4.4.5  a region the expert drew
    node_arcs          = True,   # 5.7.2.2  polygonize requires a noded input
    repair_dangling    = True,   # 5.7.1.3
    dangle_repair_px   = 1.5,
    min_face_area      = 0.0,
    build_polygons        = False,  # 5.7  False = stop at the fused lines
    diag_ignore_unlabeled = True,   # 5.8.2.5  S6/S7 skip 8001+/9001+ ids

    # 4.4.4 / 5.6  precision (F6). flatten_px must be <= fit_tol_px.
    flatten_px   = 0.05,
    grid_write   = None,      # quantisation on write only; None = full float
    svg_decimals = 4,

    # 5.8.2  comparative diagnostics. True forces one atlas rebuild PER SUBJECT.
    run_comparative_diag = True,
    metrics_avg_gl  = True,   # 5.8.2.2  M3' protrusion preservation
    metrics_curv_ks = True,   # 5.8.2.3  M7 curvature-distribution KS test
    metrics_skele   = False,  # 5.8.2.4  M4 skeleton/hull ratio
)

FUSION_GLOB        = "*.svg"    # 5.1.2  pattern matched inside FUSION_INPUT_DIR
FUSION_INPUT_PATHS = {}         # 5.1.2  optional per-subject overrides {sid: svg_path}
FUSION_WEIGHTS     = None       # 5.2.1  None = equal weights
FUSION_RUN_ID      = 1          # 5.7.4  bump each time fusion is re-run on new QA output

# =====================================================================================
# 0.4.7  MIRROR AND VOLUME                                                         [6.1]
# =====================================================================================
MIRROR_WELD_TOL     = 0.75   # 6.1.2.1  voxels; nodes this close to the midline snap onto it
MIRROR_RENDER_SCALE = 2.0    # on-screen zoom only; does not affect the saved SVG
MIRROR_STROKE       = 0.36
MIRROR_ATLAS_FILL   = 0.55   # fill opacity for the polygon atlas; 0 = lines only

# =====================================================================================
# 0.4.8  DIAGNOSTICS
# =====================================================================================
OVERLAY_INPUTS = True     # draw the per-subject inputs behind the fused line
FUSED_STROKE   = 0.36
ATLAS_FILL     = 0.55     # fill opacity for the polygon atlas; 0 = lines only

# 5.8 perturbation harness: band-limited normal-displacement fields.
PERTURB_AMP  = 1.0        # A, voxels. The dose axis.
PERTURB_K    = 8          # band limit; lambda_min = arc_length / K
PERTURB_MODE = "anti"     # {"anti","same","indep"}
PERTURB_SEED = 20260101   # fixed -> bit-reproducible; vary for replicate runs

# =====================================================================================
# 0.2  MODULES
# =====================================================================================
if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)
for _m in ("boundary_graph.py", "edge_fusion.py", "contour_fusion.py"):
    if not os.path.exists(f"{MODULES_DIR}/{_m}"):
        raise FileNotFoundError(f"{_m} not found in {MODULES_DIR}.")
import boundary_graph as bg
import contour_fusion as cf
import edge_fusion as ef

FUSION_PARAMS = ef.FusionParams(**FUSION_PARAMS_KW)   # 5.1.1

# =====================================================================================
# 0.3  RECOVER THE REGISTRATION PRODUCTS AND BUILD THE RUN TREE
# =====================================================================================
def _safe_dirname(s: str) -> str:
    """Reduce a run name to characters that are safe as a directory on Drive. [0.3.3]"""
    return re.sub(r"[^A-Za-z0-9._+-]", "_", s) if s else "default"


RUN_DIR = f"{WORK_TEST}/{_safe_dirname(RUN_PREFIX)}"   # everything this run writes
FUSION_INPUT_DIR = f"{RUN_DIR}/qc"                     # 5.1.2  per-run; no cross-run bleed
REGISTRY_PATH    = f"{RUN_DIR}/atlas/region_registry.json"   # 5.7.4


def _pfx(name: str) -> str:
    """No-op kept for the existing call sites; runs are separated by RUN_DIR. [0.3.3]"""
    return name


def _find_one(pattern, what="file"):
    """First path matching pattern; raises if nothing matches. [0.3]"""
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matched ({what}): {pattern}")
    return hits[0]


# 0.3.1  labels + MRI come from REGISTRATION_ONLY node 2.3 (subject grid, not template grid)
DB09_DIR        = f"{TPL_DIR}/template_db09"
SUBJ_DIR        = f"{WORK}/reg_subjgrid"
TEMPLATE_LABELS = _find_one(f"{SUBJ_DIR}/labels_in_subjgrid.nii.gz", "subject-grid labels")
TEMPLATE_LUT    = TEMPLATE_LABELS.replace(".nii.gz", "_LUT.csv")
TEMPLATE_T1     = _find_one(f"{SUBJ_DIR}/mri_in_subjgrid.nii.gz", "subject-grid template MRI")
SUBJ_GRID       = json.load(open(f"{SUBJ_DIR}/grid.json"))
assert os.path.exists(TEMPLATE_LUT), f"LUT missing next to labels: {TEMPLATE_LUT}"

# 0.3.2  combined_lut: id -> abbrev / name / rgb_hex / kind, rebuilt from the LUT CSV
combined_lut = {}
with open(TEMPLATE_LUT, newline="") as f:
    for row in csv.DictReader(f):
        try: rid = int(float(row["id"]))
        except (KeyError, TypeError, ValueError): continue
        combined_lut[rid] = {"abbrev": row.get("abbreviation", str(rid)),
                             "name":   row.get("name", str(rid)),
                             "rgb_hex": row.get("color_hex", ""),
                             "kind":   row.get("type", "region")}

# 0.3.3  per-run scratch tree
for d in ["state", "lines_propagated", "lines_export_for_qa/slices",
          "lines_export_for_qa/tables", "atlas", "qc"]:
    os.makedirs(f"{RUN_DIR}/{d}", exist_ok=True)


def save_state(name, obj):
    """Persist one pipeline object under RUN_DIR/state/. [0.3.3]"""
    with open(f"{RUN_DIR}/state/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)


def load_state(name, default=KeyError):
    """Lookup ladder: this run's state, then legacy flat state, then the shared work/
    cache (which holds 'regs' and 'preproc'). First hit wins. [0.3.3]"""
    cands = [f"{RUN_DIR}/state/{name}.pkl",
             f"{WORK_TEST}/state/{RUN_PREFIX}_{name}.pkl",
             f"{WORK_TEST}/state/{name}.pkl",
             f"{WORK}/state/{name}.pkl"]
    path = next((p for p in cands if os.path.exists(p)), None)
    if path is None:
        if default is KeyError:
            raise FileNotFoundError(f"state '{name}' not found (tried: {cands})")
        return default
    with open(path, "rb") as f:
        return pickle.load(f)


def report_outputs(cell_name, files=(), state_keys=(), checks=()):
    """Print the files, state keys and gate results a cell produced. [0.3.3]"""
    print(f"\n=== {cell_name}: outputs ===")
    for f in files:
        ok = os.path.exists(f); sz = os.path.getsize(f) if ok else 0
        print(f"  {'OK ' if ok and sz>0 else '!! '}{f}  ({sz} bytes)")
    for k in state_keys:
        p = f"{RUN_DIR}/state/{k}.pkl"
        print(f"  {'OK ' if os.path.exists(p) else '!! '}state/{k}.pkl")
    for label, okc, hint in checks:
        print(f"  {'PASS' if okc else 'WARN'}: {label}" + ("" if okc else f"  -> {hint}"))
    print("=" * (len(cell_name) + 16))


print("Setup done.")
print("  template labels :", TEMPLATE_LABELS)
print("  template LUT    :", TEMPLATE_LUT, f"({len(combined_lut)} regions)")
print("  template MRI    :", TEMPLATE_T1)
print("  reg cache from  :", SUBJ_DIR)
print(f"  subject grid    : {SUBJ_GRID['target_shape']} @ {SUBJ_GRID['target_zooms']} mm")
print(f"  slice index scale from template indices: x{SUBJ_GRID['slice_index_scale']:.4f}")
print("  THIS RUN writes :", RUN_DIR)
print(f"\n  parcellation: slice={SLICE_INDEX} hemi={HEMISPHERE} "
      f"simplify_tol={SIMPLIFY_TOL} smooth_sigma={SMOOTH_SIGMA if SMOOTH_BOUNDARIES else 0}")
print(f"  fusion: kp_alpha={FUSION_PARAMS.kp_alpha} fit_tol_px={FUSION_PARAMS.fit_tol_px} "
      f"rep={FUSION_PARAMS.representation} orphan={FUSION_PARAMS.orphan_policy}")
print(f"          strict_topo={FUSION_PARAMS.strict_topo} "
      f"build_polygons={FUSION_PARAMS.build_polygons} "
      f"comparative_diag={FUSION_PARAMS.run_comparative_diag}")


In [ ]:
# =====================================================================================
# 1.2  STAGE THE PREPROCESSED SCANS
#
# 1.2.1  subject ids and paths come from 0.4.2
# 1.2.3  each scan is header-checked and recorded in state 'preproc'
# =====================================================================================
import os
import nibabel as nib

# --- explicit scan paths (already skull-stripped) ------------------------------------

def import_preprocessed():
    pp = {}
    for sid, path in SUBJECT_SCANS.items():
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            raise FileNotFoundError(f"{sid}: scan not found or empty: {path}")
        nib.load(path)   # header-only sanity load
        pp[sid] = {"brain": path, "mask": None, "pre": path, "warpedtpl": None}
    save_state("preproc", pp)
    report_outputs("CELL 3 import",
                   files=[d["brain"] for d in pp.values()],
                   state_keys=["preproc"])
    print(f"\nImported {len(pp)} pre-skull-stripped subjects:")
    for sid, d in pp.items():
        print(f"  {sid}: {d['brain']}")
    return pp

import_preprocessed()


In [ ]:
# =====================================================================================
# 2.x  LOAD THE AFFINE REGISTRATION FROM CACHE   (no re-registration in test mode)
#
# Validates that the warped image and transform files REGISTRATION_ONLY wrote under
# work/reg/ still exist, then loads state 'regs'. work/ is read-only here.
# =====================================================================================
import os, pickle
import ants, nibabel as nib, numpy as np

def registration_metrics(warped_path, template_path=None):
    template_path = template_path or TEMPLATE_T1
    f = ants.image_read(template_path); m = ants.image_read(warped_path)
    fa, ma = f.numpy().ravel(), m.numpy().ravel()
    a = (fa - fa.mean())/(fa.std()+1e-9); b = (ma - ma.mean())/(ma.std()+1e-9)
    ncc = float(np.mean(a*b))
    fg = (fa > 0) | (ma > 0)
    hist,_,_ = np.histogram2d(fa[fg], ma[fg], bins=64)
    pxy = hist/(hist.sum()+1e-9); px = pxy.sum(1); py = pxy.sum(0)
    Hx = -np.sum(px[px>0]*np.log(px[px>0])); Hy = -np.sum(py[py>0]*np.log(py[py>0]))
    Hxy = -np.sum(pxy[pxy>0]*np.log(pxy[pxy>0]))
    return {"ncc": ncc, "nmi": float((Hx+Hy)/(Hxy+1e-9))}

def cell4_load_registration():
    pp = load_state("preproc")
    regs = {}
    missing = []
    for sid in pp:
        warped = f"{SUBJ_DIR}/{sid}_in_subjgrid.nii.gz"
        tfm    = f"{WORK}/reg/{sid}_transforms.pkl"
        if not (os.path.exists(warped) and os.path.exists(tfm)):
            missing.append(sid); continue
        with open(tfm, "rb") as f: saved = pickle.load(f)
        if not all(os.path.exists(p) for p in saved["fwd"] + saved["inv"]):
            missing.append(sid); continue
        regs[sid] = {**saved, "warped": warped, "seconds": 0.0, "cached": True}
    if missing:
        raise FileNotFoundError(
            f"No resampled registration for {missing} under {SUBJ_DIR}. "
            f"Run the REGISTRATION_ONLY notebook's CELL 2 then CELL 3 for these subjects.")
    save_state("regs", regs)
    checks = []
    lab_shape = nib.load(TEMPLATE_LABELS).shape
    for sid, r in regs.items():
        ncc = registration_metrics(r["warped"])["ncc"]
        checks.append((f"{sid} template overlap (NCC={ncc:.2f})", ncc > 0.5,
                       "low overlap -> re-check the full run's CELL 2 for this subject"))
        checks.append((f"{sid} on the label grid {lab_shape}",
                       nib.load(r["warped"]).shape == lab_shape,
                       "grid mismatch -> re-run the registration notebook's CELL 3"))
    report_outputs("CELL 4 load registration (cache)",
                   files=[r["warped"] for r in regs.values()],
                   state_keys=["regs"], checks=checks)
    return regs

cell4_load_registration()


In [ ]:
# =====================================================================================
# 3.1  PARCELLATION: trace ONE slice into a boundary graph
#
# 3.1.1  apply the hemisphere mask
# 3.1.2  fragment relabelling (before the voxel threshold, so arcs and seeds agree)
# 3.1.3  trace boundaries into a graph
# 3.1.4  smoothing (Gaussian low-pass) and simplification (Douglas-Peucker)
# 3.1.5  separate closed rings from open arcs
# 3.1.6  compute seeds (distance-transform maxima)
# 3.1.7  the template graph for the chosen slice is built once and copied to each subject
# =====================================================================================
def _hemisphere_mask(label_slice, hemisphere):
    """Zero out the half we are NOT keeping."""
    if hemisphere == "whole":
        return label_slice, None
    H_lr = label_slice.shape[0]
    mid = H_lr / 2.0
    out = label_slice.copy()
    rows = np.arange(H_lr)
    if hemisphere == "left":
        out[rows >= mid, :] = 0          # keep rows below midline
    elif hemisphere == "right":
        out[rows < mid, :] = 0           # keep rows at/above midline
    else:
        raise ValueError(f"HEMISPHERE must be 'whole'|'left'|'right', got {hemisphere!r}")
    return out, mid

if not (SLICE_RANGE[0] <= SLICE_INDEX <= SLICE_RANGE[1]):
    print(f"  !! note: SLICE_INDEX={SLICE_INDEX} is outside advisory SLICE_RANGE {SLICE_RANGE}; "
          f"running anyway.")

def view_template_slice(slice_index=None, save=None):
    """DEBUG HELPER: render one slice of the label volume so you can confirm SLICE_AXIS and
    see how many regions appear. Pass the slice you intend to test."""
    import matplotlib.pyplot as plt
    lab = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    if slice_index is None:
        slice_index = SLICE_INDEX
    sl = np.take(lab, slice_index, axis=SLICE_AXIS)
    print(f"slice {slice_index}: shape {sl.shape}, "
          f"{len(np.unique(sl))-1} regions present (ids {sorted(np.unique(sl))[:10]}...)")
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.rot90(sl), cmap="nipy_spectral"); ax.set_title(f"labels, slice {slice_index}")
    ax.axis("off")
    if save: fig.savefig(save, dpi=150, bbox_inches="tight")
    return slice_index

def _smooth_ring(coords, sigma):
    """Low-pass (Gaussian) smooth of a traced boundary polyline, to turn the axis-aligned pixel staircase into a real contour BEFORE it is simplified."""
    from scipy.ndimage import gaussian_filter1d
    pts = np.asarray(coords, float)
    if sigma <= 0 or len(pts) < 5:
        return pts
    closed = np.allclose(pts[0], pts[-1])
    if closed:
        ring = pts[:-1]
        x = gaussian_filter1d(ring[:, 0], sigma, mode="wrap")
        y = gaussian_filter1d(ring[:, 1], sigma, mode="wrap")
        out = np.column_stack([x, y])
        return np.vstack([out, out[0]])
    x = gaussian_filter1d(pts[:, 0], sigma, mode="nearest")
    y = gaussian_filter1d(pts[:, 1], sigma, mode="nearest")
    out = np.column_stack([x, y])
    out[0], out[-1] = pts[0], pts[-1]          # keep junction endpoints exactly put
    return out

def _simplify_to_budget(coords, tol_px, max_pts=None):
    """Douglas-Peucker to at most max_pts vertices, KEEPING THE CORNERS."""
    from shapely.geometry import LineString
    P = np.asarray(coords, float)
    if len(P) < 3:
        return P, tol_px, 0.0
    closed = np.allclose(P[0], P[-1])
    if max_pts is None or len(P) <= max_pts:
        return P, tol_px, 0.0
    lo, hi = tol_px, max(tol_px * 2, 1.0)
    ref = P
    L = LineString([tuple(p) for p in P])
    for _ in range(40):                       # find an upper tolerance that fits the budget
        if len(np.array(L.simplify(hi).coords)) <= max_pts:
            break
        hi *= 1.6
    for _ in range(24):                       # bisect down to the smallest tolerance that fits
        mid = 0.5 * (lo + hi)
        if len(np.array(L.simplify(mid).coords)) <= max_pts:
            hi = mid
        else:
            lo = mid
    Q = np.array(L.simplify(hi).coords)
    if closed and not np.allclose(Q[0], Q[-1]):
        Q = np.vstack([Q, Q[0]])
    return Q, hi, float(ef.max_deviation(ref, Q))

def label_slice_to_graph(label_slice, min_region_voxels=MIN_REGION_VOXELS,
                         simplify_tol=SIMPLIFY_TOL,
                         adaptive=True):
    """Trace boundaries from one integer label slice into a node/element graph where each element carries the TWO region codes it separates (shared borders are a SINGLE line)."""
    from scipy.ndimage import distance_transform_edt
    from shapely.geometry import LineString
    from shapely.ops import linemerge
    # one unique code per connected piece, BEFORE the voxel threshold, so arcs and the seeds
    # returned below share one fragment decomposition [edge_fusion 0b]
    lab = ef.fragment_relabel_slice(label_slice)
    H, W = lab.shape
    keep = np.zeros_like(lab)
    for rid in np.unique(lab):
        if rid == 0:
            continue
        m = lab == rid
        if m.sum() >= min_region_voxels:
            keep[m] = rid
    lab = keep

    from collections import defaultdict
    segs = defaultdict(list)
    for c in range(W - 1):
        diff = lab[:, c] != lab[:, c + 1]
        for r in np.where(diff)[0]:
            a, b = sorted((int(lab[r, c]), int(lab[r, c + 1])))
            segs[(a, b)].append([(c + 0.5, r - 0.5), (c + 0.5, r + 0.5)])
    for r in range(H - 1):
        diff = lab[r, :] != lab[r + 1, :]
        for c in np.where(diff)[0]:
            a, b = sorted((int(lab[r, c]), int(lab[r + 1, c])))
            segs[(a, b)].append([(c - 0.5, r + 0.5), (c + 0.5, r + 0.5)])

    g = bg.BoundaryGraph()          # local accumulator (was a module-level global; unsafe to reuse
                                    # now that add_node defers .nodes -- a fresh graph each call)
    n_pts, max_dev = 0, 0.0
    for (a, b), seglist in segs.items():
        merged = linemerge([LineString(s) for s in seglist])
        geoms = merged.geoms if merged.geom_type == "MultiLineString" else [merged]
        for geom in geoms:
            if SMOOTH_BOUNDARIES:
                # smooth FIRST (kills the staircase), THEN simplify ONCE at the normal tolerance.
                # DP now drops the redundant points along the smooth curve instead of re-finding
                # the step corners, because after smoothing there are no step corners left to find.
                pts = _smooth_ring(np.array(geom.coords), SMOOTH_SIGMA)
                if simplify_tol > 0 and len(pts) >= 2:
                    pts = np.array(LineString(pts).simplify(simplify_tol).coords)
            else:
                pts = np.array(geom.simplify(simplify_tol).coords)
            if adaptive and len(pts) >= 3 and TRACE_MAX_HANDLES:
                pts, _t, dev = _simplify_to_budget(pts, simplify_tol, TRACE_MAX_HANDLES)
                max_dev = max(max_dev, dev)            # cost of the handle cap, reported below
            if len(pts) < 2:
                continue
            n_pts += len(pts)
            prev = None
            for xy in pts:
                ni = g.add_node(xy)
                if prev is not None:
                    g.add_element(prev, ni, a, b)   # both codes set at creation
                prev = ni

    # The graph was built with a grid hash that DEFERS the .nodes array (add_node appends to a
    # pending list; it no longer re-vstacks on every insert, which was an O(n^2) build). Materialise
    # .nodes ONCE here, now the build loop is done.
    g.finalize_nodes()

    # ---- SEEDS: one deep interior point per region -----------------------------------
    # WHAT A SEED IS: for each region, the single voxel FARTHEST from that region's boundary --
    # the peak of the distance transform (distance_transform_edt gives every voxel its distance to
    # the nearest non-region voxel; argmax is the deepest point). So a seed is a point GUARANTEED
    # to lie well inside its region, one per region id, stored as ((x, y), region_id).
    #
    seeds = []
    for rid in np.unique(lab):
        if rid == 0:
            continue
        m = lab == rid
        dt = distance_transform_edt(m)
        iy, ix = np.unravel_index(np.argmax(dt), dt.shape)   # deepest interior voxel
        seeds.append(((float(ix), float(iy)), int(rid)))
    if max_dev > 0:
        print(f"  !! TRACE_MAX_HANDLES={TRACE_MAX_HANDLES} cost up to {max_dev:.3f}px of "
              f"boundary deviation. That is a LOSSY ergonomics choice made BEFORE QA and "
              f"BEFORE fusion. Set it to None unless Illustrator is actually unusable.")
    return g, seeds

def _seed_points_on_slice(label_slice, min_region_voxels=SEED_MIN_VOXELS):
    """Seeds = one deep interior point per region (distance-transform maximum)."""
    from scipy.ndimage import distance_transform_edt
    seeds = []
    for rid in np.unique(label_slice):
        if rid == 0:
            continue
        m = label_slice == rid
        if m.sum() < min_region_voxels:
            continue
        dt = distance_transform_edt(m)
        iy, ix = np.unravel_index(np.argmax(dt), dt.shape)
        seeds.append(((float(ix), float(iy)), int(rid)))
    return seeds

def propagate_single_slice(min_region_voxels=MIN_REGION_VOXELS,
                           simplify_tol=SIMPLIFY_TOL):
    """Build the template boundary graph for SLICE_INDEX once, then give each subject an IDENTICAL copy (template-space design -> same slice, same labels for every subject)."""
    regs = load_state("regs")
    lab = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    n_slices = lab.shape[SLICE_AXIS]
    if not (0 <= SLICE_INDEX < n_slices):
        raise IndexError(f"SLICE_INDEX={SLICE_INDEX} out of range 0..{n_slices-1}")

    sl = np.take(lab, SLICE_INDEX, axis=SLICE_AXIS)
    sl, midline_lr = _hemisphere_mask(sl, HEMISPHERE)
    n_regions = len(np.unique(sl)) - 1
    if (sl > 0).sum() < min_region_voxels:
        raise ValueError(f"slice {SLICE_INDEX} (hemisphere='{HEMISPHERE}') is essentially "
                         f"empty; pick another SLICE_INDEX or HEMISPHERE.")

    g, seeds = label_slice_to_graph(sl, min_region_voxels, simplify_tol, adaptive=True)
    template_graphs = {SLICE_INDEX: (g, seeds)}
    save_state("template_graphs", template_graphs)

    # give every subject an identical copy in template space (no per-subject warp needed)
    out = {}
    for sid in regs:
        gc = bg.BoundaryGraph(nodes=np.array(g.nodes).copy(),
                              elements=[e[:] for e in g.elements])
        outp = f"{RUN_DIR}/lines_propagated/{sid}_slice_{SLICE_INDEX:03d}.txt"
        bg.write_ascii(gc, outp)
        out[sid] = {SLICE_INDEX: outp}
    save_state("propagated", out)

    save_state("test_meta", {"slice_index": SLICE_INDEX, "label_detail": LABEL_DETAIL,
                             "line_mode": LINE_MODE, "double_offset": LINE_DOUBLE_OFFSET,
                             "trace_max_handles": TRACE_MAX_HANDLES,
                             "simplify_tol": simplify_tol,
                             "hemisphere": HEMISPHERE, "midline_lr": midline_lr})

    # F1a diagnostic: how many points did the sagitta bound actually allocate, per arc?
    arcs = ef.graph_to_arcs(g)
    npts = [len(a.pts) for a in arcs]
    checks = [
        ("template slice has regions", n_regions > 0, "pick a slice with labels"),
        (f"slice {SLICE_INDEX} node count sane ({len(g.nodes)})",
         10 < len(g.nodes) < 60000, "adjust simplify_tol"),
        ("same slice+labels copied to every subject", len(out) == len(regs),
         "a subject is missing from regs"),
        ("handles per boundary are draggable in Illustrator",
         (max(npts) <= 200 if npts else True),
         f"max {max(npts) if npts else 0} handles on one boundary -> raise simplify_tol, or set "
         f"TRACE_MAX_HANDLES (lossy)"),
    ]
    report_outputs("CELL 5 propagate (single slice, template space)",
                   files=[list(v.values())[0] for v in out.values()],
                   state_keys=["template_graphs", "propagated", "test_meta"], checks=checks)
    print(f"\nPropagated slice {SLICE_INDEX} (hemisphere='{HEMISPHERE}') to {len(out)} subjects "
          f"({n_regions} regions, {len(g.nodes)} anchor nodes, {len(arcs)} arcs, "
          f"label_detail='{LABEL_DETAIL}', line_mode='{LINE_MODE}').")
    if npts:
        print(f"  Douglas-Peucker anchors (simplify_tol={simplify_tol}px): "
              f"{min(npts)}..{max(npts)} pts/arc, median {int(np.median(npts))}, "
              f"{sum(npts)} total.")
        print(f"  The old N_BOUNDARY_ANCHORS=24 would have written {24*len(arcs)} points, "
              f"EVENLY SPACED BY ARC LENGTH -- landing them BETWEEN the corners and rounding "
              f"every notch away, before the expert ever saw it.")
    if HEMISPHERE != "whole":
        print(f"  NOTE: single hemisphere kept (midline L-R row = {midline_lr:.1f}). "
              f"Run CELL 8 after QA/fusion to mirror it back to a full slice.")
    return out

propagate_single_slice()


In [ ]:
# =====================================================================================
# 3.2 / 4.4  SVG ROUND-TRIP: export one slice for correction, and read it back
#
# 3.2.3    layered SVG: background (locked), boundaries (editable), labels
# 3.2.3.4  double-line mode writes each border twice, one path per owning region
# 3.2.4    region reference table
# 3.2.6    round-trip gate: codes must survive and geometry error stay in tolerance
# 3.2.7    the export is copied into RUN_DIR/qc/ as the fusion input
# 4.4.1    codes are recovered from id="bnd_A_B_k" (Illustrator strips data-*)
# 4.4.2    _xHH_ escapes are undone; 4.4.3 ancestor transforms composed
# 4.4.4    Bezier segments flattened at flatten_px
# 4.4.5    closed uncoded path -> new expert id (9001+); open uncoded path counted only
# 4.4.6    double-line pairs collapsed back to one line
# =====================================================================================
import base64, io, os, csv, colorsys, math
import numpy as np, nibabel as nib

# ---- display / labeling options -----------------------------------------------------

def _fmt(v):
    """Coordinate formatter. Quantisation happens ON WRITE ONLY (AFAM F6 / SATM C8) -- there
    is none anywhere inside the fusion. grid_write=None -> full float."""
    p = FUSION_PARAMS
    if p.grid_write:
        v = round(float(v) / p.grid_write) * p.grid_write
    return f"{float(v):.{p.svg_decimals}f}"

# ---- number / color / name lookups (KEPT) -------------------------------------------
def _number_regions(anchors):
    rids = sorted({a[0] for a in anchors})
    return rids, {rid: i + 1 for i, rid in enumerate(rids)}

def region_color(rid):
    rid = ef.fragment_base(int(rid))   # every fragment of a region shares ONE colour
    p = FUSION_PARAMS
    if rid >= p.unlabeled_id_start and rid < p.new_id_start:
        return (255, 0, 255)          # MAGENTA = an UNLABELED region (a QA gap). Unmissable.
    h = (rid * 0.61803398875) % 1.0; s, v = 0.65, 0.92
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (round(r*255), round(g*255), round(b*255))

def _rgb_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(*rgb)

def load_region_table(lut_csv=None):
    lut_csv = lut_csv or TEMPLATE_LUT
    table = {}
    if not (lut_csv and os.path.exists(lut_csv)):
        print(f"  !! LUT not found at {lut_csv} -- labels fall back to numeric ids.")
        return table
    with open(lut_csv, newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        print(f"  !! LUT {lut_csv} is empty."); return table
    cols = {c.lower().strip(): c for c in rows[0].keys()}
    id_col   = cols.get("id")
    abbr_col = cols.get("abbreviation") or cols.get("abbrev") or cols.get("label")
    name_col = cols.get("name") or cols.get("long_name") or cols.get("region")
    hex_col  = cols.get("color_hex") or cols.get("hex")
    for row in rows:
        try: rid = int(float(row.get(id_col)))
        except (TypeError, ValueError): continue
        abbrev = (row.get(abbr_col) if abbr_col else None) or str(rid)
        name   = (row.get(name_col) if name_col else None) or abbrev
        abbrev = str(abbrev).split(":")[-1].strip(); name = str(name).strip()
        rgb = region_color(rid)
        if hex_col and row.get(hex_col, "").startswith("#"):
            h = row[hex_col].lstrip("#")
            try: rgb = (int(h[0:2],16), int(h[2:4],16), int(h[4:6],16))
            except ValueError: pass
        table[rid] = {"abbrev": abbrev, "name": name, "rgb": rgb}
    return table

def _abbrev_for(table, rid):
    code = int(rid)
    rid, fx = ef.fragment_base(code), ef.fragment_index(code)   # name comes from the BASE
    p = FUSION_PARAMS
    if rid == 0: return "bg"
    if p.unlabeled_id_start <= rid < p.new_id_start: ab = f"UNLAB{rid}"
    elif rid >= p.new_id_start: ab = f"NEW{rid}"
    else: ab = table.get(rid, {}).get("abbrev", str(rid))
    return ab if fx == 0 else f"{ab}#{fx}"

# ---- ORIENTATION transform (coronal) -- KEPT ----------------------------------------
def _orient_image(sl):
    img = sl.T
    img = img[::-1, :]
    if FLIP_LR: img = img[:, ::-1]
    return img

def _orient_xy(x_col, y_row, H_lr, W_si):
    nr = (W_si - 1) - x_col
    nc = (H_lr - 1 - y_row) if FLIP_LR else y_row
    return nc, nr

def _unorient_xy(x_disp, y_disp, H_lr, W_si):
    nc = (H_lr - 1 - x_disp) if FLIP_LR else x_disp
    x_col = (W_si - 1) - y_disp
    y_row = nc
    return x_col, y_row

def _orient_arr(P, H_lr, W_si):
    """Vectorised voxel -> display for an (n,2) array."""
    P = np.asarray(P, float)
    nr = (W_si - 1) - P[:, 0]
    nc = ((H_lr - 1) - P[:, 1]) if FLIP_LR else P[:, 1]
    return np.column_stack([nc, nr])

def _unorient_arr(D, H_lr, W_si):
    D = np.asarray(D, float)
    nc = ((H_lr - 1) - D[:, 0]) if FLIP_LR else D[:, 0]
    x_col = (W_si - 1) - D[:, 1]
    return np.column_stack([x_col, nc])

def _slice_to_png_bytes(t1_volume, slice_index, axis,
                        hemisphere="whole", midline_lr=None):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    raw = np.take(t1_volume, slice_index, axis=axis).astype(np.float32)
    img = _orient_image(raw)               # shape (H, W); W = raw axis-0 (L-R)
    lo, hi = np.percentile(img[img > 0], (1, 99)) if (img > 0).any() else (0, 1)
    img = np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)

    if hemisphere != "whole" and midline_lr is not None:
        W_img = img.shape[1]
        mid_col = int(round(midline_lr))
        keep_low = (hemisphere == "left")
        if FLIP_LR:
            keep_low = not keep_low
            mid_col = W_img - mid_col
        if keep_low:
            img[:, mid_col:] = 0.0
        else:
            img[:, :mid_col] = 0.0

    buf = io.BytesIO(); h, w = img.shape
    fig = plt.figure(figsize=(w/100, h/100), dpi=100); ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img, cmap="gray", origin="upper"); ax.axis("off")
    fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    return buf.getvalue(), w, h, raw.shape

# ---- label placement (KEPT; eats (rid, x, y, clearance, n_instances)) ----------------
def _label_points_on_slice(label_slice, label_all_instances=None, min_component_voxels=None):
    from scipy.ndimage import distance_transform_edt, label as cc_label
    if label_all_instances is None: label_all_instances = LABEL_ALL_INSTANCES
    if min_component_voxels is None: min_component_voxels = MIN_COMPONENT_VOXELS
    # fragments carry their own codes, so each code below has exactly one component
    label_slice = ef.fragment_relabel_slice(label_slice, min_voxels=1)
    anchors = []
    for rid in np.unique(label_slice):
        if rid == 0: continue
        comps, n = cc_label(label_slice == rid)
        allc = []
        for ci in range(1, n + 1):
            comp = comps == ci
            dt = distance_transform_edt(comp)
            iy, ix = np.unravel_index(np.argmax(dt), dt.shape)
            allc.append((int(rid), float(ix), float(iy), float(dt[iy, ix]), int(comp.sum())))
        if not allc: continue
        kept = [c for c in allc if c[4] >= min_component_voxels]
        if not kept: kept = [max(allc, key=lambda t: t[4])]
        ninst = len(kept)
        if label_all_instances:
            anchors.extend([(r, x, y, c, ninst) for (r, x, y, c, _s) in kept])
        else:
            r, x, y, c, _s = max(kept, key=lambda t: t[3])
            anchors.append((r, x, y, c, ninst))
    # LABEL_ALL_FRAGMENTS=False -> ONE label per base region, on its largest fragment
    if not FUSION_PARAMS.label_all_fragments:
        anchors = ef.filter_anchors_largest_fragment(anchors)
    return anchors

def _place_labels(anchors, label_text_of, H_lr, W_si, W, H):
    from collections import defaultdict
    by_region = defaultdict(list)
    for rid, x, y, clearance, _ninst in anchors:
        by_region[rid].append((x, y, clearance))
    labels = []
    for rid, comps in by_region.items():
        txt = str(label_text_of.get(rid, rid)); ndig = max(len(txt), 1)
        for (x, y, clearance) in comps:
            d = 2.0 * clearance
            fs = min((d * 0.9) / (ndig * LABEL_FIT), d * LABEL_HEIGHT_FRAC)
            fs = min(max(fs, LABEL_FS_MIN), LABEL_FS_MAX)
            xd, yd = _orient_xy(x, y, H_lr, W_si)
            labels.append({"rid": rid, "cx": xd, "cy": yd, "fs": fs, "txt": txt})
    return labels

# SVG READER  --  the QA round-trip.  THIS IS WHERE ILLUSTRATOR BREAKS THINGS.
# data-matA / data-matB. Both assumptions fail on a real Illustrator round-trip:
#
#   * ILLUSTRATOR CONVERTS <polyline> TO <path d="M ... L ... Z">.
#     -> a polyline-only reader silently returns an EMPTY graph.
#   * ILLUSTRATOR STRIPS UNKNOWN data-* ATTRIBUTES on save.
import xml.etree.ElementTree as ET
import re as _re

def _svg_unescape(s):
    """Illustrator writes _x5F_ for '_', _x2D_ for '-', etc. Undo it."""
    return _re.sub(r"_x([0-9A-Fa-f]{2})_", lambda m: chr(int(m.group(1), 16)), s or "")

from xml.sax.saxutils import escape as _xml_escape  # stdlib: turns & < > into &amp; &lt; &gt;

def _parse_transform(t):
    """Compose an SVG transform string into a 2x3 affine [[a,c,e],[b,d,f]]."""
    M = np.array([[1., 0., 0.], [0., 1., 0.]])
    if not t:
        return M
    for name, args in _re.findall(r"(\w+)\s*\(([^)]*)\)", t):
        v = [float(x) for x in _re.split(r"[\s,]+", args.strip()) if x]
        if name == "matrix" and len(v) == 6:
            N = np.array([[v[0], v[2], v[4]], [v[1], v[3], v[5]]])
        elif name == "translate":
            N = np.array([[1., 0., v[0]], [0., 1., v[1] if len(v) > 1 else 0.]])
        elif name == "scale":
            sx = v[0]; sy = v[1] if len(v) > 1 else v[0]
            N = np.array([[sx, 0., 0.], [0., sy, 0.]])
        elif name == "rotate" and len(v) >= 1:
            a = math.radians(v[0]); c, s = math.cos(a), math.sin(a)
            N = np.array([[c, -s, 0.], [s, c, 0.]])
            if len(v) == 3:
                T1 = np.array([[1., 0., v[1]], [0., 1., v[2]]])
                T2 = np.array([[1., 0., -v[1]], [0., 1., -v[2]]])
                N = _mm(_mm(T1, N), T2)
        else:
            continue
        M = _mm(M, N)
    return M

def _mm(A, B):
    A3 = np.vstack([A, [0., 0., 1.]]); B3 = np.vstack([B, [0., 0., 1.]])
    return (A3 @ B3)[:2]

def _apply_tf(M, P):
    P = np.asarray(P, float)
    return np.column_stack([M[0, 0]*P[:, 0] + M[0, 1]*P[:, 1] + M[0, 2],
                            M[1, 0]*P[:, 0] + M[1, 1]*P[:, 1] + M[1, 2]])

def _flatten_cubic(p0, p1, p2, p3, tol):
    """Adaptive cubic flattening. Splits until the control polygon is within tol of the chord."""
    d1 = np.abs(np.cross(p3 - p0, p0 - p1)); d2 = np.abs(np.cross(p3 - p0, p0 - p2))
    L = np.hypot(*(p3 - p0)) + 1e-12
    n = max(2, int(math.ceil(math.sqrt(max(d1, d2) / L / max(tol, 1e-6) * 3.0))) + 1)
    n = min(n, 64)
    t = np.linspace(0, 1, n)[1:, None]
    return ((1-t)**3)*p0 + 3*((1-t)**2)*t*p1 + 3*(1-t)*(t**2)*p2 + (t**3)*p3

def _parse_path_d(d, tol):
    """SVG 'd' -> [(pts, closed), ...]."""
    toks = _re.findall(r"([MmLlHhVvCcSsQqTtZz])|(-?\d*\.?\d+(?:[eE][-+]?\d+)?)", d or "")
    items = [(a or b) for a, b in toks]
    subs, pts, cur, start, cmd, prev_c2, prev_q = [], [], np.zeros(2), np.zeros(2), None, None, None
    i = 0
    def _num():
        nonlocal i
        v = float(items[i]); i += 1; return v
    while i < len(items):
        if items[i] in "MmLlHhVvCcSsQqTtZz":
            cmd = items[i]; i += 1
        if cmd is None:
            i += 1; continue
        rel = cmd.islower(); C = cmd.upper()
        if C == "M":
            if pts and len(pts) >= 2:
                subs.append((np.array(pts), False))
            x, y = _num(), _num()
            cur = (cur + [x, y]) if rel else np.array([x, y])
            start = cur.copy(); pts = [cur.copy()]
            cmd = "l" if rel else "L"
        elif C == "L":
            x, y = _num(), _num()
            cur = (cur + [x, y]) if rel else np.array([x, y])
            pts.append(cur.copy())
        elif C == "H":
            x = _num(); cur = np.array([cur[0] + x, cur[1]]) if rel else np.array([x, cur[1]])
            pts.append(cur.copy())
        elif C == "V":
            y = _num(); cur = np.array([cur[0], cur[1] + y]) if rel else np.array([cur[0], y])
            pts.append(cur.copy())
        elif C in ("C", "S"):
            if C == "C":
                c1 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            else:
                c1 = 2*cur - prev_c2 if prev_c2 is not None else cur.copy()
            c2 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            p3 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            pts.extend(_flatten_cubic(cur, c1, c2, p3, tol))
            prev_c2 = c2; cur = p3; prev_q = None
        elif C in ("Q", "T"):
            if C == "Q":
                q = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            else:
                q = 2*cur - prev_q if prev_q is not None else cur.copy()
            p2 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            c1 = cur + 2.0/3.0*(q - cur); c2 = p2 + 2.0/3.0*(q - p2)
            pts.extend(_flatten_cubic(cur, c1, c2, p2, tol))
            prev_q = q; cur = p2; prev_c2 = None
        elif C == "Z":
            if len(pts) >= 2:
                subs.append((np.array(pts), True))
            pts = [start.copy()]; cur = start.copy()
            cmd = None
        else:
            i += 1
        if C not in ("C", "S"): prev_c2 = None
        if C not in ("Q", "T"): prev_q = None
    if len(pts) >= 2:
        subs.append((np.array(pts), False))
    return subs

def _meta_from_el(el):
    """(codes, arc_index, side) from data-* if the editor kept them, else from the id."""
    eid = _svg_unescape(el.get("id", ""))
    m = _re.match(r"^bnd_(-?\d+)_(-?\d+)_(\d+)(?:_s(\d+))?$", eid)
    code = arc = side = None
    if m:
        code = (int(m.group(1)), int(m.group(2)))
        arc  = int(m.group(3))
        side = int(m.group(4)) if m.group(4) is not None else None
    a, b = el.get("data-matA"), el.get("data-matB")
    if a is not None and b is not None:
        try: code = (int(a), int(b))
        except ValueError: pass
    if el.get("data-arc") is not None:
        try: arc = int(el.get("data-arc"))
        except ValueError: pass
    return code, arc, side

def _project_onto(P, R):
    """Closest point on polyline R for each point of P. Vectorised point-to-segment."""
    P = np.asarray(P, float); R = np.asarray(R, float)
    A, B = R[:-1], R[1:]
    AB = B - A
    L2 = (AB ** 2).sum(1) + 1e-12
    t = np.clip(((P[:, None, :] - A[None]) * AB[None]).sum(2) / L2[None], 0.0, 1.0)
    proj = A[None] + t[..., None] * AB[None]                 # (n, m, 2)
    d = np.linalg.norm(P[:, None, :] - proj, axis=2)         # (n, m)
    return proj[np.arange(len(P)), d.argmin(axis=1)]

def _collapse_double_lines(items):
    """LINE_MODE='double' writes TWO half-lines per border, offset +/- double_offset from the true boundary."""
    from collections import defaultdict
    by = defaultdict(list)
    for (code, arc, side, P, closed) in items:
        by[(code, arc, closed)].append((P, side))
    out, n_pairs = [], 0
    for (code, arc, closed), group in by.items():
        sides = {s for _, s in group if s is not None}
        if len(group) == 2 and len(sides) == 2:
            A, B = group[0][0], group[1][0]
            if len(A) == len(B):                      # untouched: vertex i <-> vertex i. Exact.
                if np.sum((A - B) ** 2) > np.sum((A - B[::-1]) ** 2):
                    B = B[::-1]
                M = 0.5 * (A + B)
            else:                                     # edited: project instead
                M = 0.5 * (A + _project_onto(A, B))
            out.append((code, M, closed))
            n_pairs += 1
        else:
            for P, _ in group:
                out.append((code, P, closed))
    return out, n_pairs

def read_qa_svg(svg_path, H_lr, W_si, verbose=True, line_mode=None, double_offset=None,
                params=None):
    """Parse a QA-edited SVG (Inkscape OR Illustrator) back into a BoundaryGraph carrying BOTH region codes on every element."""
    params = params or FUSION_PARAMS
    meta = load_state("test_meta", default={})
    line_mode     = line_mode     or meta.get("line_mode", "single")
    double_offset = double_offset if double_offset is not None else meta.get("double_offset", 0.35)

    tree = ET.parse(svg_path); root = tree.getroot()
    parent = {c: p for p in root.iter() for c in p}

    def _lname(t): return t.rsplit("}", 1)[-1]
    def _tf_chain(el):
        M, node = np.array([[1., 0., 0.], [0., 1., 0.]]), el
        chain = []
        while node is not None:
            chain.append(node.get("transform"))
            node = parent.get(node)
        for t in reversed(chain):
            if t: M = _mm(M, _parse_transform(t))
        return M

    items, n_new, n_nocode, n_path, n_poly, n_pairs = [], 0, 0, 0, 0, 0
    for el in root.iter():
        tag = _lname(el.tag)
        if tag not in ("polyline", "path", "polygon"):
            continue
        # skip the atlas-face layer if it is present (those are fills, not boundaries)
        if (el.get("id") or "").startswith("face_"):
            continue
        M = _tf_chain(el)

        subs = []
        if tag in ("polyline", "polygon"):
            raw = (el.get("points") or "").strip()
            if not raw:
                continue
            v = [float(t) for t in _re.split(r"[\s,]+", raw) if t]
            if len(v) < 4:
                continue
            subs = [(np.array(v, float).reshape(-1, 2), tag == "polygon")]
            n_poly += 1
        else:
            subs = _parse_path_d(el.get("d", ""), params.flatten_px)
            n_path += 1
        if not subs:
            continue

        code, arc_i, side = _meta_from_el(el)
        for si, (P, closed) in enumerate(subs):
            if len(P) < 2:
                continue
            D = _apply_tf(M, P)
            vox = _unorient_arr(D, H_lr, W_si)         # display -> voxel
            if code is None:
                # No code anywhere. A CLOSED shape is a region the EXPERT DREW -> new id.
                # An OPEN one is a stray stroke: it cannot bound anything, so it is ignored
                # (and counted, so it is never silently swallowed).
                n_nocode += 1
                if closed or (len(vox) > 3 and np.hypot(*(vox[0] - vox[-1])) < 1.0):
                    items.append((None, None, None, vox, True))
                continue
            items.append((tuple(sorted(code)), (arc_i, si), side, vox, bool(closed)))

    coded   = [it for it in items if it[0] is not None]
    uncoded = [(None, P, True) for (c, a, s, P, cl) in items if c is None]
    if line_mode == "double":
        collapsed, n_pairs = _collapse_double_lines(coded)
    else:
        collapsed = [(c, P, cl) for (c, a, s, P, cl) in coded]
    collapsed += uncoded

    # -- expert-drawn closed paths with no code -> NEW regions -------------------------
    reg = ef.load_registry(REGISTRY_PATH)
    next_new = ef.registry_next_new(reg, params)
    final = []
    for (code, P, closed) in collapsed:
        if code is None:
            code = (params.outer_code, next_new)
            next_new += 1
            n_new += 1
        final.append((tuple(sorted(code)), P, closed))

    g = bg.BoundaryGraph(nodes=np.empty((0, 2)), elements=[])
    index, xy = {}, []
    def _nid(p):
        k = (round(float(p[0]), 6), round(float(p[1]), 6))
        if k not in index:
            index[k] = len(xy); xy.append((float(p[0]), float(p[1])))
        return index[k]
    for (a, b), P, closed in final:
        Q = np.vstack([P, P[0]]) if (closed and np.hypot(*(P[0] - P[-1])) > 1e-9) else P
        prev = None
        for p in Q:
            ni = _nid(p)
            if prev is not None and prev != ni:
                g.elements.append([prev, ni, int(a), int(b)])
            prev = ni
    g.nodes = np.asarray(xy, float) if xy else np.zeros((0, 2))

    if verbose:
        print(f"    {os.path.basename(svg_path)}: {n_poly} polyline(s) + {n_path} path(s) -> "
              f"{len(g.elements)} elements, {len(ef.graph_to_arcs(g))} arcs"
              + (f", {n_pairs} double-line pair(s) collapsed" if n_pairs else "")
              + (f", {n_new} NEW expert region(s) from id {params.new_id_start}" if n_new else "")
              + (f", {n_nocode} open shape(s) with no code IGNORED" if n_nocode and not n_new else ""))
        if n_path and not n_poly:
            print(f"      (this file is Illustrator-saved: <polyline> became <path>, and the "
                  f"codes were recovered from id='bnd_A_B_k' because data-* was stripped)")
    return g

# ---- smoothing / offsetting (KEPT, but smoothing is now OFF by default) --------------
def _smooth_contour(pts, closed, sigma):
    """Gaussian-smooth a display polyline."""
    from scipy.ndimage import gaussian_filter1d
    P = np.asarray(pts, float)
    if len(P) < 4 or sigma <= 0: return P
    if closed and np.allclose(P[0], P[-1]): P = P[:-1]
    ring = np.vstack([P, P[0]]) if closed else P
    seg = np.sqrt((np.diff(ring, axis=0) ** 2).sum(1)); d = np.r_[0, np.cumsum(seg)]
    if d[-1] <= 0: return P
    u = np.arange(0, d[-1], 1.0)
    xs = np.interp(u, d, ring[:, 0]); ys = np.interp(u, d, ring[:, 1])
    if closed:
        xs = gaussian_filter1d(xs, sigma, mode="wrap"); ys = gaussian_filter1d(ys, sigma, mode="wrap")
        out = np.c_[xs, ys]; return np.vstack([out, out[0]])
    else:
        end0, end1 = ring[0].copy(), ring[-1].copy()
        xs = gaussian_filter1d(xs, sigma, mode="nearest"); ys = gaussian_filter1d(ys, sigma, mode="nearest")
        out = np.c_[xs, ys]; out[0], out[-1] = end0, end1
        return out

def _offset_polyline(disp, offset):
    """Offset an ordered display polyline by `offset` px along its per-vertex normal.
    Used to build the two half-lines in double mode. CELL 7 collapses them back."""
    P = np.asarray(disp, float)
    if len(P) < 2: return P
    closed = np.allclose(P[0], P[-1])
    Q = P[:-1] if closed else P
    n = len(Q)
    tang = np.zeros_like(Q)
    for i in range(n):
        a = Q[i-1] if (closed or i > 0) else Q[i]
        b = Q[(i+1) % n] if (closed or i < n-1) else Q[i]
        t = b - a; nrm = np.hypot(*t)
        tang[i] = t / nrm if nrm > 1e-9 else np.array([1.0, 0.0])
    normal = np.c_[-tang[:, 1], tang[:, 0]]
    out = Q + offset * normal
    if closed: out = np.vstack([out, out[0]])
    return out

# ---- SVG writer: ARC-BASED boundaries layer, codes in the id ------------------------
def _graph_to_svg(g, path, t1_volume=None, slice_index=None, axis=2,
                  label_anchors=None, table=None, stroke=1.2,
                  line_mode="single", double_offset=0.35, label_detail="abbrev",
                  hemisphere="whole", midline_lr=None, smooth_sigma=None,
                  regions=None, fill_opacity=0.0):
    """Write one slice to SVG."""
    if not len(g.nodes) or t1_volume is None or slice_index is None:
        open(path, "w").write("<svg xmlns='http://www.w3.org/2000/svg'/>"); return
    table = table or {}
    sigma = EXPORT_SMOOTH_SIGMA if smooth_sigma is None else smooth_sigma
    png, W, H, (H_lr, W_si) = _slice_to_png_bytes(
        t1_volume, slice_index, axis, hemisphere=hemisphere, midline_lr=midline_lr)
    b64 = base64.b64encode(png).decode("ascii")

    show_text = (label_detail != "none")
    region_ids, _num = _number_regions(label_anchors or [])
    abbrev_of = {rid: _abbrev_for(table, rid) for rid in region_ids}
    labels = _place_labels(label_anchors, abbrev_of, H_lr, W_si, W, H) \
             if (label_anchors and show_text) else []

    total_w, total_h = W, H
    out = ['<?xml version="1.0" encoding="UTF-8"?>',
           f'<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" '
           f'xmlns:inkscape="http://www.inkscape.org/namespaces/inkscape" '
           f'xmlns:sodipodi="http://sodipodi.sourceforge.net/DTD/sodipodi-0.0.dtd" '
           f'viewBox="0 0 {total_w:.1f} {total_h:.1f}" width="{total_w:.1f}" height="{total_h:.1f}">']

    # -- LAYER 1: background (page + T1 raster). LOCKED --------------------------------
    out.append('<g inkscape:groupmode="layer" inkscape:label="background" id="layer_background" '
               'sodipodi:insensitive="true" style="pointer-events:none">')
    out.append(f'<rect x="0" y="0" width="{total_w:.1f}" height="{total_h:.1f}" fill="white"/>')
    out.append(f'<image x="0" y="0" width="{W}" height="{H}" '
               f'xlink:href="data:image/png;base64,{b64}" style="pointer-events:none"/>')
    out.append('</g>')

    # -- LAYER 1b: the POLYGON ATLAS (CELL 7's rebuild). LOCKED. ------------------------
    # THE POLYGONS ARE THE ATLAS. Faces get their region id by INTERSECTING the codes of the
    # arcs on their boundary; whatever that rule cannot decide is an explicit UNLABELED region,
    # drawn MAGENTA, so a gap in the QA shows up ON the atlas instead of vanishing.
    if regions and fill_opacity > 0:
        out.append('<g inkscape:groupmode="layer" inkscape:label="atlas_faces" id="layer_faces" '
                   f'sodipodi:insensitive="true" style="pointer-events:none" '
                   f'opacity="{fill_opacity:.2f}">')
        for rid, faces in sorted(regions.items()):
            col = _rgb_hex(region_color(rid))
            for k, poly in enumerate(faces):
                rings = [np.asarray(poly.exterior.coords, float)] + \
                        [np.asarray(r.coords, float) for r in poly.interiors]
                d = []
                for R in rings:                       # even-odd fill -> holes are real holes
                    D = _orient_arr(R, H_lr, W_si)
                    d.append("M " + " L ".join(f"{_fmt(x)},{_fmt(y)}" for x, y in D) + " Z")
                out.append(f'<path d="{" ".join(d)}" fill="{col}" fill-rule="evenodd" '
                           f'stroke="none" id="face_{int(rid)}_{k}" data-region="{int(rid)}"/>')
        out.append('</g>')

    # -- LAYER 2: boundaries (editable). ONE POLYLINE PER **ARC**. UNLOCKED. ------------
    out.append('<g inkscape:groupmode="layer" inkscape:label="boundaries" id="layer_boundaries" '
               'fill="none" stroke-linejoin="round" stroke-linecap="round">')
    for k, arc in enumerate(ef.graph_to_arcs(g)):
        mA, mB = int(arc.code[0]), int(arc.code[1])
        P = np.vstack([arc.pts, arc.pts[0]]) if arc.closed else arc.pts
        disp = _orient_arr(P, H_lr, W_si)
        if len(disp) < 2:
            continue
        if sigma > 0:
            disp = _smooth_contour(disp, bool(arc.closed), sigma)
        na, nb = _abbrev_for(table, mA), _abbrev_for(table, mB)
        # id carries the codes. Illustrator STRIPS data-* but KEEPS id -- this is the only
        # channel that survives an Illustrator round-trip.
        eid = f"bnd_{mA}_{mB}_{k}"
        if line_mode == "double":
            half = stroke / 2.0
            colA = _rgb_hex(region_color(mA)) if mA > 0 else "#888888"
            colB = _rgb_hex(region_color(mB)) if mB > 0 else "#888888"
            out.append(f'<g id="arc_{eid}" data-arc="{k}">')
            for si, (sgn, col, mat) in enumerate(((+1.0, colA, mA), (-1.0, colB, mB))):
                od = _offset_polyline(disp, sgn * double_offset)
                pts_str = " ".join(f"{_fmt(x)},{_fmt(y)}" for (x, y) in od)
                out.append(f'<polyline id="{eid}_s{si}" points="{pts_str}" stroke="{col}" '
                           f'stroke-width="{half:.3f}" fill="none" data-matA="{mA}" '
                           f'data-matB="{mB}" data-side="{mat}" data-arc="{k}">'
                           f'<title>{_xml_escape(str(na))} | {_xml_escape(str(nb))}</title></polyline>')
            out.append('</g>')
        else:
            rid = mA if mA > 0 else mB
            col = _rgb_hex(region_color(rid)) if rid > 0 else "#888888"
            pts_str = " ".join(f"{_fmt(x)},{_fmt(y)}" for (x, y) in disp)
            out.append(f'<polyline id="{eid}" points="{pts_str}" stroke="{col}" '
                       f'stroke-width="{stroke}" fill="none" data-matA="{mA}" '
                       f'data-matB="{mB}" data-arc="{k}">'
                       f'<title>{_xml_escape(str(na))} | {_xml_escape(str(nb))}</title></polyline>')
    out.append('</g>')

    # -- LAYER 3: labels ---------------------------------------------------------------
    if labels:
        out.append('<g inkscape:groupmode="layer" inkscape:label="labels" id="layer_labels" '
                   'font-family="sans-serif" font-weight="bold" '
                   'text-anchor="middle" dominant-baseline="central">')
        _label_seen = {}
        for L in labels:
            col = _rgb_hex(region_color(L["rid"]))
            pos = f'x="{L["cx"]:.2f}" y="{L["cy"]:.2f}" font-size="{L["fs"]:.2f}"'
            _n = _label_seen.get(L["rid"], 0); _label_seen[L["rid"]] = _n + 1
            out.append(f'<g id="label_{int(L["rid"])}_{_n}" data-region="{L["rid"]}">')
            _txt = _xml_escape(str(L["txt"]))
            out.append(f'<text {pos} fill="black" stroke="black" '
                       f'stroke-width="{L["fs"]*LABEL_OUTLINE_FRAC:.3f}" '
                       f'stroke-linejoin="round">{_xml_escape(str(L["txt"]))}</text>')
            out.append(f'<text {pos} fill="{col}" data-region="{L["rid"]}">{_xml_escape(str(L["txt"]))}</text>')
            out.append('</g>')
        out.append('</g>')   # FIX: close the "labels" layer group opened above; without this the

    out.append('</svg>')
    _doc = "\n".join(out)
    try:
        import xml.etree.ElementTree as _ET
        _ET.fromstring(_doc)
    except _ET.ParseError as _e:
        _ln = getattr(_e, "position", (0, 0))[0]
        _lines = _doc.split("\n")
        _lo, _hi = max(0, _ln - 3), min(len(_lines), _ln + 2)
        print(f"[svg-debug] {path} is malformed: {_e}")
        for _j in range(_lo, _hi):
            print(f"[svg-debug]   {_j+1}: {_lines[_j]}")
    open(path, "w").write(_doc)

# ---- the cell: export just the toggled slice for each subject -----------------------
def cell6_export_single_slice(stroke=0.36):
    base = f"{RUN_DIR}/lines_export_for_qa"
    slices_dir, tables_dir = f"{base}/slices", f"{base}/tables"
    os.makedirs(slices_dir, exist_ok=True); os.makedirs(tables_dir, exist_ok=True)

    prop = load_state("propagated")
    regs = load_state("regs")
    meta = load_state("test_meta")
    table = load_region_table(TEMPLATE_LUT)
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    sidx = meta["slice_index"]
    line_mode = meta["line_mode"]; dbl = meta["double_offset"]; detail = meta["label_detail"]

    first_sid = next(iter(prop))
    written = []
    for sid, slices in prop.items():
        t1_vol = nib.load(regs[sid]["warped"]).get_fdata() if sid in regs else None
        g = bg.read_ascii(slices[sidx])
        lab_slice = np.take(lab_vol, sidx, axis=SLICE_AXIS)
        lab_slice, _ = _hemisphere_mask(lab_slice, meta["hemisphere"])  # FIX: mask labels to kept hemisphere
        anchors = _label_points_on_slice(lab_slice)
        svg = f"{slices_dir}/{sid}_slice_{sidx:03d}.svg"
        _graph_to_svg(g, svg, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                      label_anchors=anchors, table=table, stroke=stroke,
                      line_mode=line_mode, double_offset=dbl, label_detail=detail,
                      hemisphere=meta["hemisphere"], midline_lr=meta["midline_lr"])
        written.append(svg)
        # AUTO-QA: drop an identical copy into the fusion input dir (qc/) so a run can go
        # export -> fuse with no manual copy. lines_export_for_qa/ keeps the ORIGINAL reference.
        import shutil
        qa_dst = f"{RUN_DIR}/qc/{sid}_slice_{sidx:03d}.svg"
        shutil.copyfile(svg, qa_dst)
        print(f"  [auto-QA] {sid}: copied export -> {qa_dst}")
        if sid == first_sid and detail != "none":
            region_ids, _ = _number_regions(anchors)
            with open(f"{tables_dir}/slice_{sidx:03d}_regions.csv", "w", newline="") as f:
                w = csv.writer(f)
                w.writerow(["abbreviation", "name", "id", "color_hex", "type"])
                for rid in region_ids:
                    # one ROW per fragment (its own code), one NAME and COLOUR per base region
                    base_id = ef.fragment_base(rid); fx = ef.fragment_index(rid)
                    info = table.get(base_id, {})
                    ab = info.get("abbrev", str(base_id)); nm = info.get("name", ab)
                    if fx:
                        ab = f"{ab}#{fx}"; nm = f"{nm} (fragment {fx})"
                    col = _rgb_hex(info.get("rgb", region_color(rid)))
                    kind = combined_lut.get(base_id, {}).get("kind", "region")
                    w.writerow([ab, nm, rid, col, kind])

    g0 = bg.read_ascii(prop[first_sid][sidx])
    H_lr, W_si = np.take(lab_vol, sidx, axis=SLICE_AXIS).shape
    g_rt = read_qa_svg(written[0], H_lr, W_si, verbose=False)
    codes_out = {tuple(sorted((int(e[2]), int(e[3])))) for e in g0.elements}
    codes_in  = {tuple(sorted((int(e[2]), int(e[3])))) for e in g_rt.elements}
    d_rt = ef._curve_dist(ef.graph_to_arcs(g_rt), ef.graph_to_arcs(g0), 0.5)

    checks = [
        ("every region code survives the SVG round-trip", codes_out <= codes_in,
         f"missing on re-import: {sorted(codes_out - codes_in)[:6]}"),
        (f"round-trip geometry error {d_rt:.4f}px <= 2x grid",
         d_rt <= max(2 * (FUSION_PARAMS.grid_write or 0.01), 0.05),
         "EXPORT_SMOOTH_SIGMA > 0, or the display transform is not inverting"),
    ]
    report_outputs("CELL 6 export (single slice)", files=written, checks=checks)
    print(f"\nExported slice {sidx} for {len(written)} subjects to {slices_dir}")
    print(f"  line_mode='{line_mode}'  label_detail='{detail}'  "
          f"hemisphere='{meta.get('hemisphere','whole')}'  smooth_sigma={EXPORT_SMOOTH_SIGMA}")
    print(f"  {len(ef.graph_to_arcs(g0))} ARCS written, codes carried in id='bnd_A_B_k' "
          f"(Illustrator strips data-*, keeps id).")
    return written

def preview_qa_slice(sid=None, stroke=0.4, scale=2.0):
    """Render the toggled slice for one subject inline in Colab."""
    from IPython.display import HTML, display
    import tempfile
    prop = load_state("propagated"); regs = load_state("regs"); meta = load_state("test_meta")
    table = load_region_table(TEMPLATE_LUT)
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    sid = sid or next(iter(prop)); sidx = meta["slice_index"]
    g = bg.read_ascii(prop[sid][sidx])
    t1_vol = nib.load(regs[sid]["warped"]).get_fdata() if sid in regs else None
    _lab_slice, _ = _hemisphere_mask(np.take(lab_vol, sidx, axis=SLICE_AXIS),
                                     meta.get("hemisphere", "whole"))  # FIX: mask labels to kept hemisphere
    anchors = _label_points_on_slice(_lab_slice)
    tmp = tempfile.NamedTemporaryFile(suffix=".svg", delete=False).name
    _graph_to_svg(g, tmp, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                  label_anchors=anchors, table=table, stroke=stroke,
                  line_mode=meta["line_mode"], double_offset=meta["double_offset"],
                  label_detail=meta["label_detail"],
                  hemisphere=meta.get("hemisphere", "whole"),
                  midline_lr=meta.get("midline_lr"))
    svg_text = open(tmp).read()
    if svg_text.startswith("<?xml"): svg_text = svg_text.split("?>", 1)[1]
    print(f"preview subject={sid} slice={sidx} line_mode={meta['line_mode']} "
          f"label_detail={meta['label_detail']}")
    display(HTML(f'<div style="background:#888;display:inline-block;padding:6px;'
                 f'transform:scale({scale});transform-origin:top left">{svg_text}</div>'))
    return tmp

# NOTE: cell6_export_single_slice() calls read_qa_svg() for its round-trip check, and
# check; the export itself does not depend on it.
cell6_export_single_slice(stroke=0.36)
preview_qa_slice(stroke=0.4)


In [ ]:
# =====================================================================================
# 5.8  PERTURBATION HARNESS   (diagnostic input generator)
#
# Applies a band-limited displacement field along each boundary's NORMAL, so the
# perturbation is not translation-invariant and the shape metrics at 5.8.2 can see it.
# Parameters are set at 0.4.8.
# =====================================================================================
import numpy as np

prop = load_state("propagated")
sids = list(prop)
meta = load_state("test_meta"); sidx = meta["slice_index"]

# ----------------------------- TOGGLES -----------------------------------------------

def _normals_from_polyline(P, closed):
    """Unit outward-ish normals at each vertex, from the CENTRAL-DIFFERENCE tangent."""
    P = np.asarray(P, float)
    if len(P) < 2:
        return np.zeros_like(P)
    T = np.gradient(P, axis=0) if not closed else \
        (np.roll(P, -1, axis=0) - np.roll(P, 1, axis=0)) / 2.0
    L = np.hypot(T[:, 0], T[:, 1])
    L[L < 1e-12] = 1.0
    T = T / L[:, None]
    return np.c_[-T[:, 1], T[:, 0]]          # rotate tangent +90 deg

def _field(s_norm, amp, K, rng):
    """Band-limited smooth field on s in [0,1]. 1/k decay -> smooth, no high-freq spikes."""
    if amp == 0.0 or K < 1:
        return np.zeros_like(s_norm)
    a = rng.normal(size=K); b = rng.normal(size=K)
    d = np.zeros_like(s_norm)
    for k in range(1, K + 1):
        d += (a[k - 1] * np.cos(2 * np.pi * k * s_norm) +
              b[k - 1] * np.sin(2 * np.pi * k * s_norm)) / k
    m = np.max(np.abs(d))
    return amp * d / m if m > 1e-12 else d    # normalise so max|d| == amp EXACTLY

# --- per-subject signed dose so "anti" sums to zero for ANY N -------------------------
if len(sids) == 1:
    signs = [0.0]
elif PERTURB_MODE == "same":
    signs = [1.0] * len(sids)
else:                                          # anti / indep: symmetric, mean 0
    signs = list(np.linspace(+1.0, -1.0, len(sids)))

corrected_fake = {}
for j, sid in enumerate(sids):
    g = bg.read_ascii(prop[sid][sidx])
    nodes = np.asarray(g.nodes, float).copy()
    # ONE field over the whole node set, indexed by node order -> shared junctions move
    # consistently for every arc that touches them. Displacing per-arc would tear the graph.
    seed = PERTURB_SEED + (j * 7919 if PERTURB_MODE == "indep" else 0)
    rng  = np.random.default_rng(seed)
    s    = np.linspace(0.0, 1.0, len(nodes), endpoint=False)
    d    = _field(s, PERTURB_AMP, PERTURB_K, rng) * signs[j]
    N    = _normals_from_polyline(nodes, closed=False)
    g.nodes = nodes + N * d[:, None]
    corrected_fake[sid] = {sidx: g}
    print(f"  {sid}: sign {signs[j]:+.2f}  max|disp| = {np.abs(d).max():.3f} vox "   # TROUBLESHOOTING PRINT
          f"(seed {seed})")

save_state("corrected", corrected_fake)
print(f"\nPerturbed 'corrected' seeded: N={len(sids)}  A={PERTURB_AMP}  K={PERTURB_K}  "
      f"mode={PERTURB_MODE}  seed={PERTURB_SEED}")
if PERTURB_MODE == "anti":
    print("  mode=anti: signed doses sum to "
          f"{sum(signs):+.3f} -> the FUSED result must return to the UNPERTURBED template.")
save_state("perturb_meta", {"amp": PERTURB_AMP, "K": PERTURB_K,
                            "mode": PERTURB_MODE, "seed": PERTURB_SEED})


In [ ]:
# =====================================================================================
# 5  CONTOUR FUSION
#
# 5.1  setup: fusion params, per-subject boundary graphs, seeds, expected region ids
# 5.2  construct the per-subject line network (arcs + nodes, first pass)
# 5.3  inject phantom regions (point / line / tree reductions)
# 5.4  decompose again (second pass) with the phantoms in place
# 5.5  node fusion
# 5.6  per-arc fusion (edge fusion)
# 5.7  polygon construction, gated on build_polygons
# 5.8  diagnostics S1-S7
# =====================================================================================
def _resolve_fusion_inputs(sids, sidx, input_dir=None, input_paths=None):
    """One SVG path per subject."""
    input_dir   = input_dir   or FUSION_INPUT_DIR
    input_paths = input_paths if input_paths is not None else FUSION_INPUT_PATHS
    resolved = {}
    for sid in sids:
        if sid in input_paths:                                   # (1) explicit override
            p = input_paths[sid]
        elif os.path.exists(f"{input_dir}/{sid}_slice_{sidx:03d}.svg"):   # (2) this run's file
            p = f"{input_dir}/{sid}_slice_{sidx:03d}.svg"
        else:                                                    # (3) last edited
            cands = glob.glob(f"{input_dir}/*{sid}_slice_{sidx:03d}.svg")
            if not cands:
                raise FileNotFoundError(
                    f"{sid}: no fusion input SVG in {input_dir} for slice {sidx:03d} "
                    f"(run={RUN_PREFIX!r}, dir={input_dir}). Run CELL 6 (auto-QA copies "
                    f"there), or set "
                    f"FUSION_INPUT_PATHS[{sid!r}].")
            p = max(cands, key=os.path.getmtime)
            print(f"  [last-edited] {sid}: no prefix match, using newest -> {p}")
        resolved[sid] = p
        print(f"  {sid}: loading {p}")
    return resolved

def _load_fusion_graphs(sidx, input_dir=None, input_paths=None, corrected=None):
    """Get {sid: BoundaryGraph} for one slice."""
    if corrected is not None:
        return {sid: sl[sidx] for sid, sl in corrected.items() if sidx in sl}, "argument"
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    H_lr, W_si = np.take(lab_vol, sidx, axis=SLICE_AXIS).shape
    state = load_state("corrected", default=None)
    if state and all(sidx in sl for sl in state.values()):
        return {sid: sl[sidx] for sid, sl in state.items()}, "state 'corrected'"
    sids = list(load_state("propagated"))
    resolved = _resolve_fusion_inputs(sids, sidx, input_dir, input_paths)
    return {sid: read_qa_svg(p, H_lr, W_si) for sid, p in resolved.items()}, "QA SVGs on disk"

def fuse_lines(input_dir=None, input_paths=None, corrected=None, weights=None,
                     params=None, run_id=None, verbose=True):
    """Fuse the per-subject boundary graphs for the toggled slice, then REBUILD THE POLYGONS."""
    params = params or FUSION_PARAMS
    run_id = FUSION_RUN_ID if run_id is None else run_id
    weights = FUSION_WEIGHTS if weights is None else weights
    meta = load_state("test_meta"); sidx = meta["slice_index"]
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    registry = ef.load_registry(REGISTRY_PATH)

    graphs, src = _load_fusion_graphs(sidx, input_dir, input_paths, corrected)
    if len(graphs) < 1:
        raise ValueError("no subject graphs to fuse")
    print(f"  fusing {len(graphs)} subject(s) from {src}: {list(graphs)}")

    lab_slice = np.take(lab_vol, sidx, axis=SLICE_AXIS)
    lab_slice, _mid = _hemisphere_mask(lab_slice, meta.get("hemisphere", "whole"))
    # FRAGMENTS [edge_fusion 0b]: relabel AFTER masking (masking can itself split a region at
    # the midline), then put every subject's codes onto the template's canonical fragments.
    lab_slice = ef.fragment_relabel_slice(lab_slice)
    _canon = ef.fragment_canon_table(lab_slice)
    for (_s, _o, _n) in ef.canonicalize_fragments(graphs, _canon, params):
        print(f"  [fragments] subject {_s}: code {_o} -> canonical {_n}")
    seeds    = _seed_points_on_slice(lab_slice)
    expected = sorted(set(int(r) for r in np.unique(lab_slice)) - {0})

    fused_graphs, fused_regions, fused_anchors, all_diag = {}, {}, {}, {}
    import time as _time
    stage_times = {}                     # {stage: seconds}, filled by run_fusion (see below)
    _t_wall = _time.perf_counter()
    try:
        # phantom-region notes and Situation-3 hard-stops name the run, slice, and region
        # acronyms; the module has none of these, so hand them in. combined_lut is keyed by
        # BASE id, so a fragment reads as its base acronym plus "#index".
        _acr_map = {}
        for rid in expected:
            _b, _fx = ef.fragment_base(rid), ef.fragment_index(rid)
            _ab = (combined_lut.get(_b, {}) or {}).get("acronym", str(_b))
            _acr_map[int(rid)] = _ab if _fx == 0 else f"{_ab}#{_fx}"
        report_ctx = {"run_label": RUN_PREFIX, "slice_index": SLICE_INDEX, "acronyms": _acr_map}
        out = ef.run_fusion(graphs, seeds, expected, weights=weights, params=params,
                            registry=registry, run_id=run_id, report_ctx=report_ctx, slice_index=sidx,
                            verbose=verbose, timings=stage_times)
    except ef.TopologyDivergence as e:
        # The subjects' region-adjacency graphs DISAGREE. Averaging across a topology change
        # produces a locally wrong map with no warning -- SATM degrades gracefully here, the
        # edge/node method has a cliff (SATM C2). Refuse rather than emit a bad atlas.
        print(f"\n*** slice {sidx}: FUSION REFUSED ***\n{e}\n")
        raise

    registry = out["registry"]
    ef.save_registry(registry, REGISTRY_PATH)

    fused_graphs[sidx]  = (out["graph"], seeds)
    fused_node_graphs = {sidx: out["node_graph"]}
    save_state("fused_node_graphs", fused_node_graphs)
    _ng = out["node_graph"]._node_graph_report
    print(f"    node-graph hand-off        : {_ng['n_nodes']} nodes, {_ng['n_elements']} "
          f"elements (every vertex is a node)"
          + ("" if _ng["ok"] else f"  !! dup_segments={_ng['dup_segments']} "
             f"self_loops={_ng['self_loops']}"))
    fused_regions[sidx] = out["regions"]        # {rid: [Polygon]}  <-- THE ATLAS
    fused_anchors[sidx] = out["anchors"]
    all_diag[sidx]      = out["diag"]

    save_state("fused_graphs",  fused_graphs)
    save_state("fused_regions", fused_regions)
    save_state("fused_anchors", fused_anchors)
    save_state("fusion_diag",   all_diag)

    # ---- what the reviewer needs to look at ------------------------------------------
    unl = [f for f in out["flags"] if f[0] == "unlabeled"]
    mrg = [f for f in out["flags"] if f[0] == "merged_regions"]
    d = out["diag"]
    print(f"\n  slice {sidx}: {len(out['arcs'])} fused arcs -> {len(out['faces'])} faces -> "
          f"{len(out['regions'])} regions")
    print(f"    F1  interior anchors placed : {d['S4_kp_anchors_total']} "
          f"(mean {d['S4_kp_anchors_mean']}/arc)")
    print(f"    F1a points allocated        : {d['S4_pts_total']} "
          f"(a fixed N_FUSION_SAMPLES=200 would have written {200*len(out['arcs'])})")
    if "S7_budget_saturated" in d:                                      # only when build_polygons=True
        print(f"    S7_budget_saturated         : {d['S7_budget_saturated']}"
              + ("   <-- a feature your key-point extractor MISSED. Lower kp_alpha, or raise n_max_seg."
                 if d['S7_budget_saturated'] else "   (0 = every segment reached fit_tol)"))
    else:
        print("    S7_budget_saturated         : (skipped -- build_polygons=False)")
    print(f"    S3 suggested node_match_max : {d['S3_suggested_node_match_max']} "
          f"(= 3 x measured p95 displacement; you have {params.node_match_max})")
    if unl:
        print(f"\n    !! {len(unl)} UNLABELED region(s) -- a QA gap is now VISIBLE on the atlas:")
        for f in unl:
            print(f"       id {f[1]}  reason={f[2]}  seeds_inside={f[3]}  codes={f[4]}  "
                  f"area={f[5]}px^2")
    if mrg:
        print(f"    !! {len(mrg)} MERGED region pair(s) (a separating boundary is missing): "
              f"{[f[1] for f in mrg]}")

    # stage_times comes straight from run_fusion. The stages:
    #   fuse             : match nodes + key points, average every arc (F1a sampling)
    #   rebuild          : polygonize the fused arcs into the atlas faces + label them
    #   subject_rebuild  : rebuild EACH subject's own atlas (only if run_comparative_diag)
    #   diag_comparative : the S6/S7 per-region shape metrics (only if run_comparative_diag)
    #   diag_cheap       : S1..S5 pipeline-correctness checks (always; tiny)
    _wall = _time.perf_counter() - _t_wall
    print(f"\n    --- timing (slice {sidx}, {len(graphs)} subjects, {len(out['arcs'])} arcs, "
          f"{len(out['regions'])} regions) ---")
    for _stage in ("fuse", "rebuild", "subject_rebuild", "diag_comparative", "diag_cheap"):
        if _stage in stage_times:
            _sec = stage_times[_stage]
            print(f"      {_stage:18s} {_sec:7.2f}s  ({100*_sec/max(_wall,1e-9):3.0f}%)")
    print(f"      {'TOTAL run_fusion':18s} {_wall:7.2f}s")
    if not params.run_comparative_diag:
        print(f"      (run_comparative_diag=False: S6/S7 + per-subject rebuild skipped. "
              f"Set it True in FUSION_PARAMS for the shape-quality metrics.)")

    report_outputs("CELL 7 fuse (arc/node + SATM key points)",
                   files=[REGISTRY_PATH],
                   state_keys=["fused_graphs", "fused_regions", "fused_anchors", "fusion_diag"],
                   checks=[("no UNLABELED faces (no QA gaps)", not unl,
                            "look at the magenta regions in CELL 8's render"),
                           ("no merged regions", not mrg, "a separating boundary is missing"),
                           ("polygonize did not fall back", not d.get("S5_fallback_chainer", False),
                            f"polygonize error: {d.get('S5_polygonize_error')}"),
                           ("Euler identity holds (V-E+F == C)", d.get("G_euler_defect", 0) == 0,
                            "an arc, node or face was lost or duplicated")])
    return fused_graphs

fuse_lines()


In [ ]:
# =====================================================================================
# 5.7.6  RENDER THE FUSED ATLAS   (fused lines over the T1, then the filled polygons)
#
# The per-subject inputs are drawn faint behind the fused line, so a correct average
# visibly sits between its inputs. The filled polygons are the atlas.
# =====================================================================================
def _fused_underlay_volume():
    """Any subject's warped template-space image works (they share the template grid)."""
    regs = load_state("regs", default={})
    for sid, r in (regs or {}).items():
        if r.get("warped") and os.path.exists(r["warped"]):
            return nib.load(r["warped"]).get_fdata()
    return nib.load(TEMPLATE_T1).get_fdata()

def render_fused_slice_svg(overlay_inputs=OVERLAY_INPUTS, stroke=FUSED_STROKE,
                           atlas_fill=ATLAS_FILL, scale=2.0):
    from IPython.display import HTML, display
    meta   = load_state("test_meta"); sidx = meta["slice_index"]
    fused  = load_state("fused_graphs")
    if sidx not in fused:
        raise KeyError(f"no fused graph for slice {sidx}; run CELL 7 first.")
    g_fused = fused[sidx][0]
    # LINES-ONLY MODE (build_polygons=False): fused_regions/anchors are empty. Draw lines only.
    regions = load_state("fused_regions").get(sidx, {})
    anchors = load_state("fused_anchors").get(sidx, [])
    # Labels come from fused_anchors ONLY when polygons were built. With build_polygons=False
    # that list is empty, which is why the fused render lost its labels. Fall back to the SAME
    # label-anchor rule the QA export uses (_label_points_on_slice: one anchor per region
    # component at the distance-transform maximum), computed from the template label slice masked
    # to the kept hemisphere. This places a label at the deepest interior point of every region,
    # so it tracks the region as its shape moves. No polygon rebuild required.

    # BASED ON QA regions, NOT FUSED regions. They will be slightly off (unless polygons are activated)
    if not anchors:
        _lab_vol_lbl = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
        _lab_slice_lbl, _ = _hemisphere_mask(np.take(_lab_vol_lbl, sidx, axis=SLICE_AXIS),
                                             meta.get("hemisphere", "whole"))
        anchors = _label_points_on_slice(_lab_slice_lbl)
        print(f"  [labels] fused_anchors empty (build_polygons=False) -> "  # TROUBLESHOOTING PRINT
              f"placed {len(anchors)} label anchor(s) from the template label slice.")

    table   = load_region_table(TEMPLATE_LUT)
    t1_vol  = _fused_underlay_volume()

    out_svg = f"{RUN_DIR}/atlas/fused_slice_{sidx:03d}.svg"
    _graph_to_svg(g_fused, out_svg, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                  label_anchors=anchors, table=table, stroke=stroke,
                  line_mode="single",                 # the fused atlas is single-line by
                  double_offset=meta["double_offset"],#   construction: each shared border is
                  label_detail=meta["label_detail"],  #   stored and averaged EXACTLY ONCE
                  hemisphere=meta.get("hemisphere", "whole"),
                  midline_lr=meta.get("midline_lr"),
                  smooth_sigma=0.0,                   # C13: the export IS the fusion output
                  regions=regions, fill_opacity=atlas_fill)
    report_outputs("CELL 8 render fused atlas", files=[out_svg])

    svg_text = open(out_svg).read()
    if svg_text.startswith("<?xml"): svg_text = svg_text.split("?>", 1)[1]

    if overlay_inputs:
        graphs, src = _load_fusion_graphs(sidx)
        png, W, H, (H_lr, W_si) = _slice_to_png_bytes(t1_vol, sidx, SLICE_AXIS)
        overlay = [f'<g fill="none" stroke="#222" stroke-width="{stroke*0.6:.3f}" opacity="0.5">']
        for sid, gi in graphs.items():
            for arc in ef.graph_to_arcs(gi):
                P = np.vstack([arc.pts, arc.pts[0]]) if arc.closed else arc.pts
                D = _orient_arr(P, H_lr, W_si)
                if len(D) < 2: continue
                overlay.append('<polyline points="%s"/>' %
                               " ".join(f"{x:.2f},{y:.2f}" for x, y in D))
        overlay.append('</g>')
        gt = svg_text.find(">")
        svg_text = svg_text[:gt+1] + "\n" + "\n".join(overlay) + svg_text[gt+1:]

    unl = sorted(r for r in regions
                 if FUSION_PARAMS.unlabeled_id_start <= r < FUSION_PARAMS.new_id_start)
    new = sorted(r for r in regions if r >= FUSION_PARAMS.new_id_start)
    print(f"fused atlas: slice={sidx}  {len(regions)} regions, "
          f"{sum(len(f) for f in regions.values())} faces  (fill={atlas_fill})")
    print(f"  bold color = the fused line;  dark thin = the per-subject inputs")
    if unl: print(f"  !! MAGENTA regions {unl} are UNLABELED -- QA gaps, visible by design")
    if new: print(f"  expert-drawn NEW regions: {new}")
    display(HTML(f'<div style="background:#888;display:inline-block;padding:6px;'
                 f'transform:scale({scale});transform-origin:top left">{svg_text}</div>'))
    return out_svg

def print_fusion_report(verbose=False):
    """The S1..S7 table for the toggled slice."""
    meta = load_state("test_meta"); sidx = meta["slice_index"]
    d = load_state("fusion_diag")[sidx]
    return ef.print_report(d, f"slice {sidx}", verbose=verbose)

def print_fusion_params(p=None):
    """Echo the exact FusionParams this fused output was produced with, so the render and the
    numbers above it are self-documenting. See the parameter reference doc for what each means."""
    import dataclasses as _dc
    p = p or FUSION_PARAMS
    print("\n=== FUSION PARAMETERS used for this fused output ===")
    print(f"  RUN_PREFIX = {RUN_PREFIX!r}   FUSION_RUN_ID = {FUSION_RUN_ID}   "
          f"weights = {FUSION_WEIGHTS or 'equal'}")
    print(f"  RUN_DIR    = {RUN_DIR}")
    for f in _dc.fields(p):
        print(f"    {f.name:22s} = {getattr(p, f.name)!r}")
    print("=" * 52)

render_fused_slice_svg()
print_fusion_report()
print_fusion_params()


In [ ]:
# =====================================================================================
# 6.1  MIRROR THE HEMISPHERE BACK FOR BILATERAL SYMMETRY
#
# 6.1.1  reflect the kept-hemisphere graph across the L-R midline
# 6.1.2  weld the reflection to the original: merge seam nodes, drop duplicate arcs
# 6.1.3  rebuild full-brain faces from the welded graph
# 6.1.5  fragments are merged back to their base region ids first
#
# Does nothing when HEMISPHERE == 'whole'.
# =====================================================================================
def mirror_hemisphere_graph(g, midline_lr, weld_tol=MIRROR_WELD_TOL):
    """Reflect a single-hemisphere boundary graph across the L-R midline and union it with the
    original. Node coords are (x=col=SI, y=row=LR); the L-R (y) coordinate is mirrored."""
    arcs = ef.graph_to_arcs(g)
    out = []
    for a in arcs:
        out.append(a)
        P = np.asarray(a.pts, float).copy()
        P[:, 1] = 2.0 * midline_lr - P[:, 1]
        near = np.abs(P[:, 1] - midline_lr) <= weld_tol
        P[near, 1] = midline_lr                       # snap to the axis so it welds exactly
        out.append(ef.Arc(a.code, P, a.closed, sid="mirror"))
    # weld the two halves into ONE graph (exact float coincidence only; no grid)
    for a in out:
        P = np.asarray(a.pts, float)
        near = np.abs(P[:, 1] - midline_lr) <= weld_tol
        P[near, 1] = midline_lr
        a.pts = P
    return ef.arcs_to_graph(out)

def cell7_mirror_back(source="fused", rebuild_polygons=None):
    # rebuild_polygons=None -> follow FUSION_PARAMS.build_polygons (default).
    if rebuild_polygons is None:
        rebuild_polygons = FUSION_PARAMS.build_polygons
    """source='fused' -> CELL 7's fused graph (default). 'corrected' -> a re-imported QA graph."""
    meta = load_state("test_meta")
    hemi = meta.get("hemisphere", "whole")
    sidx = meta["slice_index"]

    if source == "fused":
        graphs = {s: gv[0] for s, gv in load_state("fused_graphs").items()}
    elif source == "corrected":
        corr = load_state("corrected")
        first = next(iter(corr))
        graphs = dict(corr[first])
    else:
        raise ValueError("source must be 'fused' or 'corrected'")

    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    atlas_full, atlas_regions = {}, {}

    if hemi == "whole":
        print("HEMISPHERE='whole' -> nothing to mirror; passing graphs through unchanged.")
        for s, g in graphs.items():
            outp = f"{RUN_DIR}/atlas/atlas_full_slice_{s:03d}.txt"
            bg.write_ascii(g, outp); atlas_full[s] = outp
            # fragments are ONE region in the finished atlas (base ids); {} when
            # build_polygons=False
            atlas_regions[s] = ef.merge_fragment_regions(
                load_state("fused_regions").get(s, {}))
        save_state("atlas_lines_full", atlas_full)
        save_state("atlas_regions_full", atlas_regions)
        report_outputs("CELL 9 mirror-back (pass-through)", files=list(atlas_full.values()),
                       state_keys=["atlas_lines_full", "atlas_regions_full"])
        return atlas_full

    midline = meta["midline_lr"]
    for s, g in graphs.items():
        full = mirror_hemisphere_graph(g, midline)
        outp = f"{RUN_DIR}/atlas/atlas_full_slice_{s:03d}.txt"
        bg.write_ascii(full, outp)
        atlas_full[s] = outp
        print(f"slice {s:03d}: mirrored {len(g.nodes)} -> {len(full.nodes)} nodes "
              f"(hemisphere='{hemi}', midline L-R={midline:.1f})")

        if rebuild_polygons:
            # REBUILD THE POLYGONS on the welded full slice. Mirroring is a topology change --
            # the two halves now share the midline -- so the faces must be re-derived, not
            # mirrored. Region ids are bilaterally symmetric, so a region present on both
            # sides simply gets TWO faces, which region_anchors then labels twice.
            # Seeds come from the SAME relabelled half-slice the fusion used, then mirrored:
            # the fragment decomposition of the bilateral slice is NOT the one the arcs carry.
            sl, _mid = _hemisphere_mask(np.take(lab_vol, s, axis=SLICE_AXIS), hemi)
            sl = ef.fragment_relabel_slice(sl)
            seeds = _seed_points_on_slice(sl)
            seeds = seeds + [((x, 2.0 * midline - y), rid) for ((x, y), rid) in seeds]
            rb = ef.rebuild_atlas(ef.graph_to_arcs(full), seeds, FUSION_PARAMS)
            atlas_regions[s] = ef.merge_fragment_regions(rb["regions"])
            unl = [f for f in rb["flags"] if f[0] == "unlabeled"]
            print(f"           rebuilt: {len(rb['faces'])} faces -> {len(rb['regions'])} regions"
                  + (f"  !! {len(unl)} UNLABELED" if unl else ""))

    if not rebuild_polygons:
            print("  [build_polygons=False] mirrored LINES only; no polygon rebuild. "  # TROUBLESHOOTING PRINT
                  "atlas_regions_full is empty by design.")
    save_state("atlas_lines_full", atlas_full)
    save_state("atlas_regions_full", atlas_regions)

    g_half = graphs[sidx]; g_full = bg.read_ascii(atlas_full[sidx])
    yv = np.asarray(g_full.nodes, float)[:, 1]
    n_seam = int((np.abs(yv - midline) < 1e-6).sum())
    # NOTE on the weld check: counting graph COMPONENTS would be wrong here -- an island is a
    # closed loop and is therefore its own component BY DESIGN, so a correctly welded slice does
    # not have one component. What actually matters is that the two halves SHARE their midline
    # count is STRICTLY LESS than twice the half's.
    welded = len(g_full.nodes) < 2 * len(g_half.nodes)
    checks = [
        ("full slice ~2x half nodes",
         len(g_full.nodes) >= 1.6 * len(g_half.nodes) - 2, "mirror did not duplicate nodes"),
        ("nodes present on BOTH sides of midline",
         (yv < midline).any() and (yv > midline).any(),
         "reflection axis wrong -> check midline_lr / HEMISPHERE"),
        (f"the two halves are WELDED at the midline ({n_seam} shared seam nodes, "
         f"{2*len(g_half.nodes) - len(g_full.nodes)} vertices merged)",
         welded and n_seam > 0,
         "raise MIRROR_WELD_TOL: the seam nodes are not landing on identical coordinates, so "
         "the halves are two disjoint graphs that merely touch, and polygonize will not close "
         "faces across the midline"),
    ]
    report_outputs("CELL 9 mirror-back", files=list(atlas_full.values()),
                   state_keys=["atlas_lines_full", "atlas_regions_full"], checks=checks)
    return atlas_full

# MIRRORED RENDER  --  same on-screen treatment CELL 6 and CELL 8 get, plus an SVG on disk
# Reuses _graph_to_svg (CELL 6) and _fused_underlay_volume (CELL 8) unchanged. Two arguments
# differ from CELL 8's call, and both are deliberate:
#
#   hemisphere="whole", midline_lr=None
#       _slice_to_png_bytes BLANKS the discarded half of the T1 underlay whenever it is given

def render_mirrored_slice_svg(atlas_full=None, stroke=MIRROR_STROKE,
                              atlas_fill=MIRROR_ATLAS_FILL, scale=MIRROR_RENDER_SCALE):
    """Render the mirrored (bilateral) slice inline AND write it to RUN_DIR/atlas/."""
    from IPython.display import HTML, display
    meta = load_state("test_meta"); sidx = meta["slice_index"]
    hemi = meta.get("hemisphere", "whole")

    if atlas_full is None:
        atlas_full = load_state("atlas_lines_full")
    if sidx not in atlas_full:
        raise KeyError(f"no mirrored graph for slice {sidx}; run cell7_mirror_back() first.")

    g_full  = bg.read_ascii(atlas_full[sidx])
    # empty when build_polygons=False -- lines-only render, exactly like CELL 8's lines mode
    regions = load_state("atlas_regions_full", default={}).get(sidx, {})
    table   = load_region_table(TEMPLATE_LUT)
    t1_vol  = _fused_underlay_volume()

    _lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    anchors  = _label_points_on_slice(np.take(_lab_vol, sidx, axis=SLICE_AXIS))
    print(f"  [mirror-render] hemisphere='{hemi}' -> underlay drawn WHOLE; "
          f"{len(anchors)} label anchor(s) from the FULL (unmasked) template label slice.")

    out_svg = f"{RUN_DIR}/atlas/atlas_full_slice_{sidx:03d}.svg"
    _graph_to_svg(g_full, out_svg, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                  label_anchors=anchors, table=table, stroke=stroke,
                  line_mode="single",                  # mirrored atlas is single-line, as fused
                  double_offset=meta["double_offset"],
                  label_detail=meta["label_detail"],
                  hemisphere="whole",                  # <<< see the note above
                  midline_lr=None,                     # <<< ditto
                  smooth_sigma=0.0,                    # C13: the export IS the mirror output
                  regions=regions, fill_opacity=atlas_fill)
    report_outputs("CELL 9 mirrored render", files=[out_svg])

    svg_text = open(out_svg).read()
    if svg_text.startswith("<?xml"):
        svg_text = svg_text.split("?>", 1)[1]

    unl = sorted(r for r in regions
                 if FUSION_PARAMS.unlabeled_id_start <= r < FUSION_PARAMS.new_id_start)
    print(f"mirrored atlas: slice={sidx}  {len(g_full.nodes)} nodes, {len(regions)} regions, "
          f"{sum(len(f) for f in regions.values())} faces  (fill={atlas_fill})")
    if unl:
        print(f"  !! MAGENTA regions {unl} are UNLABELED -- QA gaps, visible by design")
    if not regions:
        print("  [build_polygons=False] LINES ONLY; no filled faces to draw.")
    display(HTML(f'<div style="background:#888;display:inline-block;padding:6px;'
                 f'transform:scale({scale});transform-origin:top left">{svg_text}</div>'))
    return out_svg

_atlas_full = cell7_mirror_back(source="fused")
render_mirrored_slice_svg(_atlas_full)


In [ ]:
# =====================================================================================
# 5.8.3 / 5.8.4  VERIFICATION
#
# 5.8.3  S7_sweep_identity: all weight on subject i must return subject i's own map,
#        within fit_tol*3 + node_tol/2. Needs no reference data.
# 5.8.4  M8 weight sweep: the fused output tracked across the weight simplex.
# =====================================================================================
from scipy.spatial import cKDTree
import numpy as np

meta = load_state("test_meta"); sidx = meta["slice_index"]
lab_vol   = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
lab_slice = np.take(lab_vol, sidx, axis=SLICE_AXIS)
lab_slice, _m = _hemisphere_mask(lab_slice, meta.get("hemisphere", "whole"))
seeds    = _seed_points_on_slice(lab_slice)
expected = sorted(set(int(r) for r in np.unique(lab_slice)) - {0})
graphs, src = _load_fusion_graphs(sidx)
print(f"verifying on {len(graphs)} subject(s) from {src}\n")

# This is the VERIFICATION cell: it inspects the S6/S7 shape metrics, so it needs the comparative
# diagnostics ON even if FUSION_PARAMS has them OFF for the fast QA pass. Use VERIFY_PARAMS (a
# diagnostics-forced copy) wherever a metric value is read back.
import dataclasses as _dc0
VERIFY_PARAMS = ef.FusionParams(**{**_dc0.asdict(FUSION_PARAMS),
                                   "run_comparative_diag": True, "metrics_avg_gl": True})

# --- 1) S7_sweep_identity  (AFAM priority 2: the free regression test) ---------------
si = ef.sweep_identity(graphs, FUSION_PARAMS)
print("=== S7_sweep_identity ===")
print(f"  worst deviation      : {si['S7_sweep_identity_px']:.4f} px")
print(f"  bound (fit_tol*3 + node_tol/2) : {si['S7_sweep_identity_bound_px']:.4f} px")
print(f"  per subject          : {si['S7_sweep_identity_per_subject']}")
print(f"  {'PASS' if si['S7_sweep_identity_ok'] else 'FAIL'}")
d0 = load_state("fusion_diag")[sidx]
if d0.get("S7_budget_saturated", 0) > 0:
    print(f"  !! S7_budget_saturated = {d0['S7_budget_saturated']}: some segments hit "
          f"n_max_seg={FUSION_PARAMS.n_max_seg} while still over fit_tol. A feature your "
          f"key-point extractor MISSED. Lower kp_alpha (currently {FUSION_PARAMS.kp_alpha}) "
          f"or raise n_max_seg -- but lowering kp_alpha is usually the right fix.")

# --- 2) M8: weight-sweep stability curve (the paper's Fig. 4a analogue) ---------------
# REQUIRES POLYGONS: every metric in this sweep (peri_dev, round_dev, avg_gl, IoU, topo) is an
# S6/S7 shape metric, which only exists when build_polygons=True. Disabled otherwise.
if not FUSION_PARAMS.build_polygons:
    print("\n=== M8 weight sweep: SKIPPED (build_polygons=False) ===")
    print("  This curve reads S6/S7 shape metrics, which need the polygon rebuild.")
    print("  Set build_polygons=True in FUSION_PARAMS to enable it.")
elif len(graphs) >= 2:
    pair = (list(graphs)[0], list(graphs)[-1])
    print(f"\n=== M8 weight sweep: {pair[0]} -> {pair[1]} ===")
    print(f"  {'w':>5s} {'peri_dev':>10s} {'round_dev':>10s} {'avg_gl':>9s} "
          f"{'IoU':>6s} {'topo':>5s} {'regions':>8s}")
    for row in ef.sweep_weights(graphs, seeds, expected, pair=pair, steps=11,
                                params=VERIFY_PARAMS):
        if "error" in row:
            print(f"  {row['w']:5.2f}  ERROR: {row['error']}")
            continue
        print(f"  {row['w']:5.2f} {row.get('S7_peri_dev_mean_signed', 0):+10.4f} "
              f"{row.get('S7_round_dev_max', 0):10.4f} "
              f"{row.get('S7_avg_gl_dev_mean', 0):+9.4f} "
              f"{str(row.get('S6_iou_area_weighted')):>6s} "
              f"{str(row.get('S7_topo_delta')):>5s} {row.get('n_regions', 0):8d}")
    print("  peri_dev should bottom out at w=0.5 (maximum averaging) and return toward 0 at")
    print("  the ends. topo must be 0 at every w -- it is a design guarantee, not a metric.")

# S7_peri_dev on ONE real slice, key points ON vs OFF. If OFF is already near zero, intra-edge
# REQUIRES POLYGONS: peri_dev / avg_gl / round are S7 metrics, so this table only runs when
# build_polygons=True. (S4 anchor counts alone would run, but the table's point is the S7 columns.)
if not FUSION_PARAMS.build_polygons:
    print(f"\n=== does F1 earn its keep on this slice? SKIPPED (build_polygons=False) ===")
    print("  Its peri_dev/avg_gl/round columns are S7 shape metrics; enable build_polygons=True.")
else:
    print(f"\n=== does F1 earn its keep on this slice? (AFAM priority 1) ===")
    print(f"  {'config':38s} {'peri_dev':>10s} {'avg_gl':>9s} {'round':>8s} {'anchors':>8s} {'sat':>4s}")
import dataclasses as _dc
for tag, kw in ((f"keypoints OFF (= Karcher mean)",           dict(keypoints=False)),
                (f"keypoints ON, kp_alpha={FUSION_PARAMS.kp_alpha}",
                                                              dict(keypoints=True)),
                (f"keypoints ON, kp_alpha={FUSION_PARAMS.kp_alpha/2}",
                                                              dict(keypoints=True,
                                                                   kp_alpha=FUSION_PARAMS.kp_alpha/2,
                                                                   curv_sigma_px=None))):
    # This test READS S7 metrics, so it needs the comparative diagnostics ON regardless of what
    # FUSION_PARAMS is set to for the fast QA pass. avg_gl is one of the columns printed.
    p = ef.FusionParams(**{**_dc.asdict(FUSION_PARAMS), **kw, "strict_topo": False,
                           "run_comparative_diag": True, "metrics_avg_gl": True})
    try:
        o = ef.run_fusion(graphs, seeds, expected, params=p, slice_index=sidx, verbose=False)
        dd = o["diag"]
        print(f"  {tag:38s} {dd['S7_peri_dev_mean_signed']:+10.4f} "
              f"{dd['S7_avg_gl_dev_mean']:+9.4f} {dd['S7_round_dev_max']:8.4f} "
              f"{dd['S4_kp_anchors_total']:8d} {dd['S7_budget_saturated']:4d}")
    except Exception as e:
        print(f"  {tag:38s}  ERROR {type(e).__name__}: {e}")
print("  A kp_alpha COARSER than your smallest feature is WORSE THAN NO KEY POINTS AT ALL")
print("  (measured on a 6px notch: alpha=8 halved it; alpha=4 preserved it). Tune alpha FIRST.")

# --- 4) LEGACY baseline: the synthetic circle, via contour_fusion ---------------------
# every shared border twice and drops islands.
#def circle(cx=12, cy=0, r=10, n=200):
    #t = np.linspace(0, 2*np.pi, n, endpoint=False)
    #return np.c_[cx + r*np.cos(t), cy + r*np.sin(t)]
#c = circle(); c_rolled = np.roll(c, 50, axis=0)


In [ ]:
# =====================================================================================
# 5.7.4  REGION REGISTRY   (persistent, PER SLICE)
#
# A region legitimately does not appear on most slices, so the registry is keyed by
# slice and 'absent' is measured against what the template says should be on that slice.
# 5.7.4.1  fragment codes are rolled up to their base region id.
# =====================================================================================
reg = ef.load_registry(REGISTRY_PATH)
P = FUSION_PARAMS

if not reg["slices"]:
    if not FUSION_PARAMS.build_polygons:
        print(f"registry empty ({REGISTRY_PATH}) -- EXPECTED with build_polygons=False.")
        print("  The registry only tracks regions once polygons are built. Set build_polygons=True")
        print("  to populate it. Nothing is wrong here.")
    else:
        print(f"registry empty ({REGISTRY_PATH}) -- run CELL 7 first.")
else:
    print(f"{'slice':>6s} {'active':>7s} {'template':>9s} {'expert':>18s} "
          f"{'UNLABELED (QA gaps)':>22s} {'inactive':>20s}")
    for s in sorted(int(k) for k in reg["slices"]):
        regs = reg["slices"][str(s)]["regions"]
        act  = {int(r): v for r, v in regs.items() if v.get("status") == "active"}
        tpl  = sorted(r for r in act if r < P.unlabeled_id_start)
        unl  = sorted(r for r in act if P.unlabeled_id_start <= r < P.new_id_start)
        new  = sorted(r for r in act if r >= P.new_id_start)
        ina  = ef.registry_inactive(reg, s)
        print(f"{s:6d} {len(act):7d} {len(tpl):9d} {str(new):>18s} "
              f"{str(unl):>22s} {str(ina[:6]):>20s}")
        _bs = ef.registry_base_status(reg, s) # Potential source of error
        _gone = sorted(b for b, st in _bs.items() if st != "active")
        if _gone:
            print(f"        base rollup: {_gone} have NO active fragment on this slice; "
                  f"only such a base may EVER be treated as deleted")
    print(f"\n  next_new_id = {reg.get('next_new_id')}   "
          f"next_unlabeled_id = {reg.get('next_unlabeled_id')}")
    print(f"  registry file: {REGISTRY_PATH}")

# --- housekeeping (MANUAL ONLY) -------------------------------------------------------
# A region absent from one slice is normal, not an error -- do NOT purge on a schedule.
# reg, n = ef.registry_purge(reg, slice_index=None)   # hard-delete every inactive record
# ef.save_registry(reg, REGISTRY_PATH)
# DELETED is a BASE-level verdict: purge a base only when registry_base_status reports it inactive on every slice — individual fragment records going inactive is normal partial coverage.

# --- what to do about an UNLABELED region --------------------------------------------
# a separating boundary is missing or two regions merged. The fix is in the QA, not here:
#   1. CELL 8's render shows it MAGENTA. Find it.
#   2. CELL 7 printed its `seeds_inside` -- those are the region ids that merged.
#   3. Redraw the missing boundary between them in Illustrator, re-export, re-run CELL 7.
# permanent id: it already has one (8001+), and the registry now tracks it.
